# 16 - LaMa Difference Maps

## Truth Source

Origin: existing Notebook 13.

Planned additions:

- standardize spatial-error maps;
- standardize signed-improvement maps;
- add mask and boundary overlays;
- use comparable scales;
- prepare XAI and case-report assets.

## Batch 1 Scope

This first batch establishes the project setup, helper API contract, path contract, and required input validation. It intentionally does not generate difference-map figures yet.


## Batch 1 - Setup And Input Contract

The notebook consumes the accepted LaMa restoration metadata and Notebook 15 classical metrics. Canonical downstream CSVs are written under `data/processed/metrics`, while Notebook 16 visual/report artifacts are written under `outputs/16_lama_difference_maps` and `outputs/reports/16_lama_difference_maps`.


In [1]:
from pathlib import Path


def find_project_root(start_path: Path) -> Path:
    """Find the repository root from a notebook or working directory."""
    start_path = start_path.resolve()
    root_markers = (
        ".git",
        "src",
        "data",
        "outputs",
    )

    for candidate_path in (start_path, *start_path.parents):
        marker_count = sum(
            (candidate_path / marker_name).exists()
            for marker_name in root_markers
        )

        if marker_count >= 3:
            return candidate_path

    raise FileNotFoundError(
        "Could not locate the project root from "
        f"{start_path}."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

print("Project root:", PROJECT_ROOT)


Project root: D:\Masters\FH\Thesis\painting-restoration-eval


In [2]:
from __future__ import annotations

import json
import platform
import shutil
import sys
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from time import perf_counter
from typing import Any

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
import skimage
import yaml
from IPython.display import Image as IPythonImage
from IPython.display import display
from PIL import Image


SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


from restoration_eval import error_maps
from restoration_eval.error_maps import (
    apply_mask_to_map,
    bbox_from_binary_mask,
    build_boundary_ring,
    compute_absolute_error_map,
    compute_error_map_summary,
    compute_global_visualization_scales,
    compute_signed_improvement_map,
    create_error_map_figure,
    create_spatial_overlay,
    generate_error_map_figures_for_cases,
    load_mask_bool,
    load_rgb_array,
    validate_error_map_manifest,
)


required_error_map_exports = (
    "apply_mask_to_map",
    "bbox_from_binary_mask",
    "build_boundary_ring",
    "compute_absolute_error_map",
    "compute_error_map_summary",
    "compute_global_visualization_scales",
    "compute_signed_improvement_map",
    "create_error_map_figure",
    "create_spatial_overlay",
    "generate_error_map_figures_for_cases",
    "load_mask_bool",
    "load_rgb_array",
    "validate_error_map_manifest",
)

missing_error_map_exports = [
    export_name
    for export_name in required_error_map_exports
    if not hasattr(error_maps, export_name)
]

if missing_error_map_exports:
    raise ImportError(
        "The error-map helper is missing required exports: "
        f"{missing_error_map_exports}"
    )

runtime_environment_df = pd.DataFrame(
    [
        {"component": "python", "version": platform.python_version()},
        {"component": "platform", "version": platform.platform()},
        {"component": "numpy", "version": np.__version__},
        {"component": "pandas", "version": pd.__version__},
        {"component": "Pillow", "version": PIL.__version__},
        {"component": "matplotlib", "version": matplotlib.__version__},
        {"component": "scikit-image", "version": skimage.__version__},
        {
            "component": "error_maps_module",
            "version": getattr(error_maps, "ERROR_MAP_VERSION", "unversioned"),
        },
    ]
)

display(runtime_environment_df)


,component,version
0,python,3.12.6
1,platform,Windows-11-10.0.26200-SP0
2,numpy,1.26.4
3,pandas,2.3.3
4,Pillow,9.5.0
5,matplotlib,3.11.0
6,scikit-image,0.24.0
7,error_maps_module,unversioned


In [3]:
def resolve_project_path(path_value: Any) -> Path:
    """Resolve a project-relative or absolute path."""
    if path_value is None:
        raise ValueError("Path value is None.")

    try:
        if pd.isna(path_value):
            raise ValueError("Path value is null.")
    except TypeError:
        pass

    path_text = str(path_value).strip()

    if not path_text:
        raise ValueError("Path value is blank.")

    path = Path(path_text)

    if path.is_absolute():
        return path.resolve()

    return (PROJECT_ROOT / path).resolve()


def project_relative_path(path_value: Any) -> str:
    """Return a portable project-relative path where possible."""
    path = resolve_project_path(path_value)

    try:
        return path.relative_to(PROJECT_ROOT.resolve()).as_posix()
    except ValueError:
        return path.as_posix()


def file_sha256(file_path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Calculate a SHA-256 checksum for a file."""
    digest = sha256()

    with Path(file_path).open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def build_check(check: str, observed: Any, expected: Any, passed: bool) -> dict[str, Any]:
    """Build one standardized validation record."""
    return {
        "check": check,
        "observed": observed,
        "expected": expected,
        "passed": bool(passed),
    }


def atomic_write_csv(dataframe: pd.DataFrame, output_path: Path) -> None:
    """Write a CSV through a temporary file and atomically replace output."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")

    dataframe.to_csv(temporary_path, index=False)

    if not temporary_path.is_file() or temporary_path.stat().st_size <= 0:
        raise RuntimeError(
            "Temporary CSV export is missing or empty: "
            f"{temporary_path}"
        )

    temporary_path.replace(output_path)


In [4]:
CONFIG_PATH = PROJECT_ROOT / "config" / "experiment_50_config.yaml"

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH}")

with CONFIG_PATH.open("r", encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

paths_cfg = config.get("paths", {})
preprocessing_cfg = config.get("preprocessing", {})
experiment_cfg = config.get("experiment", {})

NOTEBOOK_NAME = "16_lama_difference_maps"
MODEL_NAME = "lama"
RUN_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()

TARGET_SIZE = int(
    preprocessing_cfg.get(
        "output_size",
        experiment_cfg.get("image_size", 768),
    )
)
MASK_BINARY_THRESHOLD = 0
MASK_BBOX_MARGIN = 8
BOUNDARY_WIDTH_PIXELS = 3
BOUNDARY_MODE = "both"
PROGRESS_EVERY = 25

SPATIAL_SCHEMA_VERSION = "1.0.0"
SPATIAL_IMPLEMENTATION_NAME = "restoration_eval.error_maps"

REQUIRED_DATASETS = (
    "canonical",
    "damage_size",
)

OPTIONAL_DATASETS = (
    "mask_robustness",
    "synthetic_degradation",
)

ABSOLUTE_ERROR_PERCENTILE = 99.5
SIGNED_IMPROVEMENT_PERCENTILE = 99.5
SCALE_SAMPLE_REGION = "masked"
SCALE_RANDOM_SEED = 42
MAX_SCALE_SAMPLES_PER_CASE = 5_000

GENERATE_ALL_CASE_FIGURES = True
INCLUDE_ZERO_CONTROL_FIGURES = True
FIGURE_DPI = 150

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_DIR = PROCESSED_DIR / "metadata"
PROCESSED_METRICS_DIR = PROCESSED_DIR / "metrics"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
STAGE_OUTPUT_DIR = OUTPUTS_DIR / NOTEBOOK_NAME
STAGE_FIGURES_DIR = STAGE_OUTPUT_DIR / "figures"
ALL_CASE_FIGURES_DIR = STAGE_FIGURES_DIR / "all_cases"
SELECTED_CASE_FIGURES_DIR = STAGE_FIGURES_DIR / "selected_cases"
SUMMARY_FIGURES_DIR = STAGE_FIGURES_DIR / "summary"
REPORTS_DIR = OUTPUTS_DIR / "reports" / NOTEBOOK_NAME

RESTORATION_METADATA_PATH = METADATA_DIR / "metadata_restored_lama.csv"
CLEAN_METADATA_PATH = METADATA_DIR / "metadata_processed_clean.csv"
CLASSICAL_METRICS_PATH = PROJECT_ROOT / paths_cfg.get("metrics_dir", "outputs/metrics") / "classical_metrics_lama.csv"
NOTEBOOK_15_DIAGNOSTIC_CASES_PATH = CLASSICAL_METRICS_PATH.parent / "diagnostics" / "lama_diagnostic_cases.csv"

SPATIAL_DIAGNOSTICS_OUTPUT_PATH = PROCESSED_METRICS_DIR / "spatial_diagnostics_lama.csv"
VISUALIZATION_SCALES_OUTPUT_PATH = PROCESSED_METRICS_DIR / "spatial_diagnostic_scales_lama.csv"
CASE_ASSETS_OUTPUT_PATH = PROCESSED_METRICS_DIR / "spatial_diagnostic_case_assets_lama.csv"

SELECTED_FIGURE_MANIFEST_PATH = STAGE_OUTPUT_DIR / "lama_difference_map_manifest_selected.csv"
ALL_FIGURE_MANIFEST_PATH = STAGE_OUTPUT_DIR / "lama_difference_map_manifest_all.csv"
SPATIAL_VALIDATION_OUTPUT_PATH = STAGE_OUTPUT_DIR / "lama_difference_map_validation.csv"

STAGE_MANIFEST_JSON_PATH = REPORTS_DIR / "lama_difference_maps_manifest.json"
STAGE_REPORT_PATH = REPORTS_DIR / "lama_difference_maps_report.md"

for output_directory in (
    PROCESSED_METRICS_DIR,
    STAGE_OUTPUT_DIR,
    STAGE_FIGURES_DIR,
    ALL_CASE_FIGURES_DIR,
    SELECTED_CASE_FIGURES_DIR,
    SUMMARY_FIGURES_DIR,
    REPORTS_DIR,
):
    output_directory.mkdir(parents=True, exist_ok=True)

configuration_df = pd.DataFrame(
    [
        {"parameter": "notebook_name", "value": NOTEBOOK_NAME},
        {"parameter": "model_name", "value": MODEL_NAME},
        {"parameter": "target_size", "value": TARGET_SIZE},
        {"parameter": "mask_binary_threshold", "value": "> 0"},
        {"parameter": "mask_bbox_margin", "value": MASK_BBOX_MARGIN},
        {"parameter": "boundary_width_pixels", "value": BOUNDARY_WIDTH_PIXELS},
        {"parameter": "boundary_mode", "value": BOUNDARY_MODE},
        {"parameter": "absolute_error_percentile", "value": ABSOLUTE_ERROR_PERCENTILE},
        {"parameter": "signed_improvement_percentile", "value": SIGNED_IMPROVEMENT_PERCENTILE},
        {"parameter": "scale_sample_region", "value": SCALE_SAMPLE_REGION},
        {"parameter": "scale_random_seed", "value": SCALE_RANDOM_SEED},
        {"parameter": "maximum_scale_samples_per_case", "value": MAX_SCALE_SAMPLES_PER_CASE},
        {"parameter": "generate_all_case_figures", "value": GENERATE_ALL_CASE_FIGURES},
        {"parameter": "include_zero_control_figures", "value": INCLUDE_ZERO_CONTROL_FIGURES},
        {"parameter": "figure_dpi", "value": FIGURE_DPI},
        {"parameter": "required_datasets", "value": list(REQUIRED_DATASETS)},
        {"parameter": "optional_datasets", "value": list(OPTIONAL_DATASETS)},
    ]
)

path_contract_df = pd.DataFrame(
    [
        {"artifact": "LaMa restoration metadata", "path": RESTORATION_METADATA_PATH, "required_input": True, "planned_output": False},
        {"artifact": "Processed clean metadata", "path": CLEAN_METADATA_PATH, "required_input": True, "planned_output": False},
        {"artifact": "Notebook 15 LaMa classical metrics", "path": CLASSICAL_METRICS_PATH, "required_input": True, "planned_output": False},
        {"artifact": "Notebook 15 diagnostic cases", "path": NOTEBOOK_15_DIAGNOSTIC_CASES_PATH, "required_input": False, "planned_output": False},
        {"artifact": "Canonical spatial diagnostics", "path": SPATIAL_DIAGNOSTICS_OUTPUT_PATH, "required_input": False, "planned_output": True},
        {"artifact": "Canonical visualization scales", "path": VISUALIZATION_SCALES_OUTPUT_PATH, "required_input": False, "planned_output": True},
        {"artifact": "Canonical case asset index", "path": CASE_ASSETS_OUTPUT_PATH, "required_input": False, "planned_output": True},
        {"artifact": "Selected figure manifest", "path": SELECTED_FIGURE_MANIFEST_PATH, "required_input": False, "planned_output": True},
        {"artifact": "All-case figure manifest", "path": ALL_FIGURE_MANIFEST_PATH, "required_input": False, "planned_output": True},
        {"artifact": "Spatial validation CSV", "path": SPATIAL_VALIDATION_OUTPUT_PATH, "required_input": False, "planned_output": True},
        {"artifact": "Stage manifest JSON", "path": STAGE_MANIFEST_JSON_PATH, "required_input": False, "planned_output": True},
        {"artifact": "Stage report Markdown", "path": STAGE_REPORT_PATH, "required_input": False, "planned_output": True},
    ]
)

path_contract_df["path"] = path_contract_df["path"].map(project_relative_path)

display(configuration_df)
display(path_contract_df)


,parameter,value
0,notebook_name,16_lama_difference_maps
1,model_name,lama
2,target_size,768
3,mask_binary_threshold,> 0
4,mask_bbox_margin,8
5,boundary_width_pixels,3
6,boundary_mode,both
7,absolute_error_percentile,99.5
8,signed_improvement_percentile,99.5
9,scale_sample_region,masked


,artifact,path,required_input,planned_output
0,LaMa restoration metadata,data/processed/metadata/metadata_restored_lama...,True,False
1,Processed clean metadata,data/processed/metadata/metadata_processed_cle...,True,False
2,Notebook 15 LaMa classical metrics,outputs/metrics/classical_metrics_lama.csv,True,False
3,Notebook 15 diagnostic cases,outputs/metrics/diagnostics/lama_diagnostic_ca...,False,False
4,Canonical spatial diagnostics,data/processed/metrics/spatial_diagnostics_lam...,False,True
5,Canonical visualization scales,data/processed/metrics/spatial_diagnostic_scal...,False,True
6,Canonical case asset index,data/processed/metrics/spatial_diagnostic_case...,False,True
7,Selected figure manifest,outputs/16_lama_difference_maps/lama_differenc...,False,True
8,All-case figure manifest,outputs/16_lama_difference_maps/lama_differenc...,False,True
9,Spatial validation CSV,outputs/16_lama_difference_maps/lama_differenc...,False,True


In [5]:
required_input_paths = {
    "lama_restoration_metadata": RESTORATION_METADATA_PATH,
    "processed_clean_metadata": CLEAN_METADATA_PATH,
    "notebook_15_classical_metrics": CLASSICAL_METRICS_PATH,
}

optional_input_paths = {
    "notebook_15_diagnostic_cases": NOTEBOOK_15_DIAGNOSTIC_CASES_PATH,
}

input_inventory_rows = []

for input_name, input_path in required_input_paths.items():
    input_inventory_rows.append(
        {
            "input_name": input_name,
            "input_path": project_relative_path(input_path),
            "required": True,
            "exists": input_path.is_file(),
            "file_size_bytes": input_path.stat().st_size if input_path.is_file() else 0,
        }
    )

for input_name, input_path in optional_input_paths.items():
    input_inventory_rows.append(
        {
            "input_name": input_name,
            "input_path": project_relative_path(input_path),
            "required": False,
            "exists": input_path.is_file(),
            "file_size_bytes": input_path.stat().st_size if input_path.is_file() else 0,
        }
    )

input_inventory_df = pd.DataFrame(input_inventory_rows)

missing_required_inputs_df = input_inventory_df.loc[
    input_inventory_df["required"].astype(bool)
    & ~input_inventory_df["exists"].astype(bool)
].copy()

empty_required_inputs_df = input_inventory_df.loc[
    input_inventory_df["required"].astype(bool)
    & input_inventory_df["exists"].astype(bool)
    & input_inventory_df["file_size_bytes"].eq(0)
].copy()

input_inventory_checks_df = pd.DataFrame(
    [
        build_check(
            "all_required_inputs_exist",
            len(missing_required_inputs_df),
            0,
            missing_required_inputs_df.empty,
        ),
        build_check(
            "all_required_inputs_nonempty",
            len(empty_required_inputs_df),
            0,
            empty_required_inputs_df.empty,
        ),
    ]
)

display(input_inventory_df)
display(input_inventory_checks_df)

if not missing_required_inputs_df.empty:
    display(missing_required_inputs_df[["input_name", "input_path"]])

if not empty_required_inputs_df.empty:
    display(empty_required_inputs_df[["input_name", "input_path"]])

if not input_inventory_checks_df["passed"].astype(bool).all():
    raise RuntimeError("Notebook 16 required input inventory failed.")


,input_name,input_path,required,exists,file_size_bytes
0,lama_restoration_metadata,data/processed/metadata/metadata_restored_lama...,True,True,1055479
1,processed_clean_metadata,data/processed/metadata/metadata_processed_cle...,True,True,30443
2,notebook_15_classical_metrics,outputs/metrics/classical_metrics_lama.csv,True,True,2530737
3,notebook_15_diagnostic_cases,outputs/metrics/diagnostics/lama_diagnostic_ca...,False,True,11024


,check,observed,expected,passed
0,all_required_inputs_exist,0,0,True
1,all_required_inputs_nonempty,0,0,True


In [7]:
restoration_metadata_df = pd.read_csv(RESTORATION_METADATA_PATH)
clean_metadata_df = pd.read_csv(CLEAN_METADATA_PATH)
classical_metrics_df = pd.read_csv(CLASSICAL_METRICS_PATH)

if NOTEBOOK_15_DIAGNOSTIC_CASES_PATH.is_file():
    notebook_15_diagnostic_cases_df = pd.read_csv(NOTEBOOK_15_DIAGNOSTIC_CASES_PATH)
else:
    notebook_15_diagnostic_cases_df = pd.DataFrame()


def has_usable_identity_column(dataframe: pd.DataFrame, column_name: str) -> bool:
    if column_name not in dataframe.columns:
        return False

    values = dataframe[column_name].astype("string").str.strip()
    return values.notna().any() and values.ne("").any()


metric_identity_source = None

for candidate_column in ["metric_case_id", "restoration_case_id", "case_id"]:
    if has_usable_identity_column(classical_metrics_df, candidate_column):
        metric_identity_source = candidate_column
        break

if metric_identity_source is not None:
    if "metric_case_id" not in classical_metrics_df.columns:
        classical_metrics_df["metric_case_id"] = classical_metrics_df[metric_identity_source]
    else:
        metric_case_values = classical_metrics_df["metric_case_id"].astype("string").str.strip()
        missing_metric_case_mask = metric_case_values.isna() | metric_case_values.eq("")

        if missing_metric_case_mask.any():
            classical_metrics_df.loc[
                missing_metric_case_mask,
                "metric_case_id",
            ] = classical_metrics_df.loc[
                missing_metric_case_mask,
                metric_identity_source,
            ]

required_restoration_columns = {
    "restoration_case_id",
    "dataset_name",
    "case_id",
    "painting_id",
    "model_name",
    "clean_path",
    "damaged_path",
    "mask_path",
    "restored_path",
    "status",
    "issue",
}

required_clean_columns = {
    "painting_id",
    "content_x_min",
    "content_y_min",
    "content_x_max",
    "content_y_max",
}

required_metric_columns = {
    "metric_case_id",
    "restoration_case_id",
    "dataset_name",
    "painting_id",
    "model_name",
    "evaluation_region",
    "mse_improvement",
    "mae_improvement",
    "psnr_improvement",
    "ssim_improvement",
    "status",
    "issue",
}

missing_restoration_columns = sorted(
    required_restoration_columns - set(restoration_metadata_df.columns)
)
missing_clean_columns = sorted(
    required_clean_columns - set(clean_metadata_df.columns)
)
missing_metric_columns = sorted(
    required_metric_columns - set(classical_metrics_df.columns)
)

input_table_summary_df = pd.DataFrame(
    [
        {
            "table_name": "restoration_metadata",
            "rows": len(restoration_metadata_df),
            "columns": len(restoration_metadata_df.columns),
        },
        {
            "table_name": "clean_metadata",
            "rows": len(clean_metadata_df),
            "columns": len(clean_metadata_df.columns),
        },
        {
            "table_name": "classical_metrics",
            "rows": len(classical_metrics_df),
            "columns": len(classical_metrics_df.columns),
        },
        {
            "table_name": "notebook_15_diagnostic_cases_optional",
            "rows": len(notebook_15_diagnostic_cases_df),
            "columns": len(notebook_15_diagnostic_cases_df.columns),
        },
    ]
)

input_schema_checks_df = pd.DataFrame(
    [
        build_check(
            "restoration_metadata_nonempty",
            len(restoration_metadata_df),
            "> 0",
            len(restoration_metadata_df) > 0,
        ),
        build_check(
            "clean_metadata_nonempty",
            len(clean_metadata_df),
            "> 0",
            len(clean_metadata_df) > 0,
        ),
        build_check(
            "classical_metrics_nonempty",
            len(classical_metrics_df),
            "> 0",
            len(classical_metrics_df) > 0,
        ),
        build_check(
            "restoration_required_columns",
            missing_restoration_columns,
            [],
            not missing_restoration_columns,
        ),
        build_check(
            "clean_required_columns",
            missing_clean_columns,
            [],
            not missing_clean_columns,
        ),
        build_check(
            "classical_metric_identity_source",
            metric_identity_source,
            "metric_case_id/restoration_case_id/case_id",
            metric_identity_source is not None,
        ),
        build_check(
            "metric_required_columns",
            missing_metric_columns,
            [],
            not missing_metric_columns,
        ),
    ]
)

display(input_table_summary_df)
display(input_schema_checks_df)

if not input_schema_checks_df["passed"].astype(bool).all():
    raise RuntimeError("Notebook 16 input schema validation failed.")

print("Metric identity source:", metric_identity_source)
print("Notebook 16 input schemas validated.")

,table_name,rows,columns
0,restoration_metadata,410,218
1,clean_metadata,50,47
2,classical_metrics,2260,52
3,notebook_15_diagnostic_cases_optional,12,25


,check,observed,expected,passed
0,restoration_metadata_nonempty,410,> 0,True
1,clean_metadata_nonempty,50,> 0,True
2,classical_metrics_nonempty,2260,> 0,True
3,restoration_required_columns,[],[],True
4,clean_required_columns,[],[],True
5,classical_metric_identity_source,restoration_case_id,metric_case_id/restoration_case_id/case_id,True
6,metric_required_columns,[],[],True


Metric identity source: restoration_case_id
Notebook 16 input schemas validated.


In [8]:
def normalize_string_series(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip()


def normalize_status_value(value: object) -> str:
    if pd.isna(value):
        return "missing"

    status_text = str(value).strip().lower()

    if status_text in {"ok", "success", "succeeded", "complete", "completed"}:
        return "ok"

    if status_text in {"", "none", "nan", "<na>"}:
        return "missing"

    return status_text


def normalize_region_value(value: object) -> str:
    if pd.isna(value):
        return "missing"

    return str(value).strip().lower().replace(" ", "_").replace("-", "_")


def resolve_case_path_column(dataframe: pd.DataFrame, column_name: str) -> pd.Series:
    return dataframe[column_name].map(lambda path_value: resolve_project_path(path_value).as_posix())


lama_cases_raw_df = restoration_metadata_df.copy()
clean_metadata_normalized_df = clean_metadata_df.copy()
classical_metrics_normalized_df = classical_metrics_df.copy()

identifier_columns = [
    "restoration_case_id",
    "metric_case_id",
    "case_id",
    "painting_id",
    "dataset_name",
    "model_name",
    "mask_type",
    "damage_type",
    "damage_level",
    "damage_setting",
]

for dataframe in [
    lama_cases_raw_df,
    clean_metadata_normalized_df,
    classical_metrics_normalized_df,
]:
    for column_name in identifier_columns:
        if column_name in dataframe.columns:
            dataframe[column_name] = normalize_string_series(dataframe[column_name])

lama_cases_raw_df["restoration_status_normalized"] = lama_cases_raw_df["status"].map(
    normalize_status_value
)
classical_metrics_normalized_df["metric_status_normalized"] = classical_metrics_normalized_df[
    "status"
].map(normalize_status_value)
classical_metrics_normalized_df["evaluation_region_normalized"] = classical_metrics_normalized_df[
    "evaluation_region"
].map(normalize_region_value)

if "metric_case_id" not in classical_metrics_normalized_df.columns:
    classical_metrics_normalized_df["metric_case_id"] = classical_metrics_normalized_df[
        "restoration_case_id"
    ]

path_columns = [
    "clean_path",
    "damaged_path",
    "mask_path",
    "restored_path",
]

for path_column in path_columns:
    lama_cases_raw_df[f"{path_column}_project_relative"] = lama_cases_raw_df[path_column].map(
        project_relative_path
    )
    lama_cases_raw_df[path_column] = resolve_case_path_column(lama_cases_raw_df, path_column)
    lama_cases_raw_df[f"{path_column}_exists"] = lama_cases_raw_df[path_column].map(
        lambda path_text: Path(path_text).is_file()
    )

lama_cases_raw_df["model_name_normalized"] = lama_cases_raw_df["model_name"].str.lower()
lama_cases_raw_df["is_lama_case"] = lama_cases_raw_df["model_name_normalized"].eq(MODEL_NAME)
lama_cases_raw_df["all_required_files_exist"] = lama_cases_raw_df[
    [f"{path_column}_exists" for path_column in path_columns]
].all(axis=1)

display(
    lama_cases_raw_df[
        [
            "restoration_case_id",
            "dataset_name",
            "case_id",
            "painting_id",
            "model_name",
            "restoration_status_normalized",
            "all_required_files_exist",
        ]
    ].head()
)

print("Raw LaMa restoration rows:", len(lama_cases_raw_df))

,restoration_case_id,dataset_name,case_id,painting_id,model_name,restoration_status_normalized,all_required_files_exist
0,lama__canonical__canonical__p001_loss_large,canonical,canonical__p001_loss_large,p001,lama,ok,True
1,lama__canonical__canonical__p001_loss_small,canonical,canonical__p001_loss_small,p001,lama,ok,True
2,lama__canonical__canonical__p001_mixed_damage,canonical,canonical__p001_mixed_damage,p001,lama,ok,True
3,lama__canonical__canonical__p001_scratch_thin,canonical,canonical__p001_scratch_thin,p001,lama,ok,True
4,lama__canonical__canonical__p001_zero_control,canonical,canonical__p001_zero_control,p001,lama,ok,True


Raw LaMa restoration rows: 410


In [9]:
content_bbox_columns = [
    "content_x_min",
    "content_y_min",
    "content_x_max",
    "content_y_max",
]

optional_clean_lookup_columns = [
    "category",
    "title",
    "artist",
    "source_dataset",
    "clean_width",
    "clean_height",
    "processed_width",
    "processed_height",
]

clean_lookup_columns = [
    "painting_id",
    *content_bbox_columns,
    *[
        column_name
        for column_name in optional_clean_lookup_columns
        if column_name in clean_metadata_normalized_df.columns
    ],
]

clean_lookup_df = (
    clean_metadata_normalized_df[clean_lookup_columns]
    .drop_duplicates(subset=["painting_id"])
    .copy()
)

duplicate_clean_painting_count = clean_metadata_normalized_df.duplicated(
    subset=["painting_id"]
).sum()

columns_to_drop_before_clean_merge = [
    column_name
    for column_name in clean_lookup_columns
    if column_name != "painting_id" and column_name in lama_cases_raw_df.columns
]

lama_cases_enriched_df = lama_cases_raw_df.drop(
    columns=columns_to_drop_before_clean_merge,
    errors="ignore",
).merge(
    clean_lookup_df,
    on="painting_id",
    how="left",
    validate="many_to_one",
)

for bbox_column in content_bbox_columns:
    lama_cases_enriched_df[bbox_column] = pd.to_numeric(
        lama_cases_enriched_df[bbox_column],
        errors="coerce",
    )

lama_cases_enriched_df["has_content_bbox"] = lama_cases_enriched_df[
    content_bbox_columns
].notna().all(axis=1)

lama_cases_enriched_df["content_bbox_valid"] = (
    lama_cases_enriched_df["has_content_bbox"]
    & lama_cases_enriched_df["content_x_min"].lt(lama_cases_enriched_df["content_x_max"])
    & lama_cases_enriched_df["content_y_min"].lt(lama_cases_enriched_df["content_y_max"])
)

clean_merge_checks_df = pd.DataFrame(
    [
        build_check(
            "clean_lookup_unique_paintings",
            int(duplicate_clean_painting_count),
            0,
            duplicate_clean_painting_count == 0,
        ),
        build_check(
            "all_cases_have_content_bbox",
            int(lama_cases_enriched_df["has_content_bbox"].sum()),
            len(lama_cases_enriched_df),
            bool(lama_cases_enriched_df["has_content_bbox"].all()),
        ),
        build_check(
            "all_content_bboxes_valid",
            int(lama_cases_enriched_df["content_bbox_valid"].sum()),
            len(lama_cases_enriched_df),
            bool(lama_cases_enriched_df["content_bbox_valid"].all()),
        ),
    ]
)

display(clean_merge_checks_df)

if not clean_merge_checks_df["passed"].astype(bool).all():
    display(
        lama_cases_enriched_df.loc[
            ~lama_cases_enriched_df["content_bbox_valid"],
            [
                "restoration_case_id",
                "dataset_name",
                "case_id",
                "painting_id",
                *content_bbox_columns,
            ],
        ].head(20)
    )
    raise RuntimeError("Clean metadata enrichment failed.")

print("Enriched LaMa case table shape:", lama_cases_enriched_df.shape)
display(lama_cases_enriched_df.head())

,check,observed,expected,passed
0,clean_lookup_unique_paintings,0,0,True
1,all_cases_have_content_bbox,410,410,True
2,all_content_bboxes_valid,410,410,True


Enriched LaMa case table shape: (410, 234)


,restoration_case_id,model_name,iopaint_model_name,iopaint_package_version,lama_model_version,restoration_method,inference_mode,restoration_generator_name,restoration_generator_version,execution_device,...,content_y_min,content_x_max,content_y_max,category,title,artist,processed_width,processed_height,has_content_bbox,content_bbox_valid
0,lama__canonical__canonical__p001_loss_large,lama,lama,NaN,iopaint_lama,iopaint_lama,model_inference,restoration_eval.restoration_lama,2.0.0,cuda,...,0,715,768,portrait_figure,Juan de Pareja,Diego Velázquez,768,768,True,True
1,lama__canonical__canonical__p001_loss_small,lama,lama,NaN,iopaint_lama,iopaint_lama,model_inference,restoration_eval.restoration_lama,2.0.0,cuda,...,0,715,768,portrait_figure,Juan de Pareja,Diego Velázquez,768,768,True,True
2,lama__canonical__canonical__p001_mixed_damage,lama,lama,NaN,iopaint_lama,iopaint_lama,model_inference,restoration_eval.restoration_lama,2.0.0,cuda,...,0,715,768,portrait_figure,Juan de Pareja,Diego Velázquez,768,768,True,True
3,lama__canonical__canonical__p001_scratch_thin,lama,lama,NaN,iopaint_lama,iopaint_lama,model_inference,restoration_eval.restoration_lama,2.0.0,cuda,...,0,715,768,portrait_figure,Juan de Pareja,Diego Velázquez,768,768,True,True
4,lama__canonical__canonical__p001_zero_control,lama,lama,NaN,iopaint_lama,zero_control_copy,copied_zero_control,restoration_eval.restoration_lama,2.0.0,cuda,...,0,715,768,portrait_figure,Juan de Pareja,Diego Velázquez,768,768,True,True


In [11]:
masked_region_aliases = {
    "masked",
    "mask",
    "masked_region",
    "damage_mask",
    "damaged_mask",
}

masked_metrics_df = classical_metrics_normalized_df.loc[
    classical_metrics_normalized_df["evaluation_region_normalized"].isin(masked_region_aliases)
    & classical_metrics_normalized_df["metric_status_normalized"].eq("ok")
].copy()

if masked_metrics_df.empty:
    raise RuntimeError(
        "No successful masked-region rows found in Notebook 15 classical metrics."
    )

metric_handoff_columns = [
    "metric_case_id",
    "restoration_case_id",
    "evaluation_region",
    "mse_improvement",
    "mae_improvement",
    "psnr_improvement",
    "ssim_improvement",
]

optional_metric_handoff_columns = [
    "mse_restored",
    "mse_damaged",
    "mae_restored",
    "mae_damaged",
    "psnr_restored",
    "psnr_damaged",
    "ssim_restored",
    "ssim_damaged",
]

metric_handoff_columns.extend(
    [
        column_name
        for column_name in optional_metric_handoff_columns
        if column_name in masked_metrics_df.columns
    ]
)

masked_metric_duplicate_case_count = masked_metrics_df.duplicated(
    subset=["restoration_case_id"]
).sum()

masked_metric_lookup_df = (
    masked_metrics_df[metric_handoff_columns]
    .sort_values(["restoration_case_id", "metric_case_id"])
    .drop_duplicates(subset=["restoration_case_id"], keep="first")
    .rename(
        columns={
            "evaluation_region": "metric_evaluation_region",
            "mse_improvement": "masked_mse_improvement",
            "mae_improvement": "masked_mae_improvement",
            "psnr_improvement": "masked_psnr_improvement",
            "ssim_improvement": "masked_ssim_improvement",
        }
    )
)

lama_cases_with_metrics_df = lama_cases_enriched_df.merge(
    masked_metric_lookup_df,
    on="restoration_case_id",
    how="left",
    validate="one_to_one",
)

lama_cases_with_metrics_df["has_masked_metric"] = lama_cases_with_metrics_df[
    "masked_mse_improvement"
].notna()

metric_merge_checks_df = pd.DataFrame(
    [
        build_check(
            "masked_metric_rows_available",
            len(masked_metrics_df),
            "> 0",
            len(masked_metrics_df) > 0,
        ),
        build_check(
            "masked_metric_duplicate_restoration_ids",
            int(masked_metric_duplicate_case_count),
            0,
            masked_metric_duplicate_case_count == 0,
        ),
        build_check(
            "lama_cases_with_masked_metric_observed",
            int(lama_cases_with_metrics_df["has_masked_metric"].sum()),
            f"<= {len(lama_cases_with_metrics_df)}",
            int(lama_cases_with_metrics_df["has_masked_metric"].sum()) <= len(lama_cases_with_metrics_df),
        ),
    ]
)

display(metric_merge_checks_df)

if masked_metric_duplicate_case_count != 0:
    raise RuntimeError("Duplicate masked metric rows found for restoration_case_id.")

print(
    "Masked metric coverage:",
    int(lama_cases_with_metrics_df["has_masked_metric"].sum()),
    "/",
    len(lama_cases_with_metrics_df),
)
display(lama_cases_with_metrics_df.head())

,check,observed,expected,passed
0,masked_metric_rows_available,360,> 0,True
1,masked_metric_duplicate_restoration_ids,0,0,True
2,lama_cases_with_masked_metric_observed,360,<= 410,True


Masked metric coverage: 360 / 410


,restoration_case_id,model_name,iopaint_model_name,iopaint_package_version,lama_model_version,restoration_method,inference_mode,restoration_generator_name,restoration_generator_version,execution_device,...,processed_height,has_content_bbox,content_bbox_valid,metric_case_id,metric_evaluation_region,masked_mse_improvement,masked_mae_improvement,masked_psnr_improvement,masked_ssim_improvement,has_masked_metric
0,lama__canonical__canonical__p001_loss_large,lama,lama,NaN,iopaint_lama,iopaint_lama,model_inference,restoration_eval.restoration_lama,2.0.0,cuda,...,768,True,True,lama__canonical__canonical__p001_loss_large,masked_region,48373.386292,209.135288,19.357618,NaN,True
1,lama__canonical__canonical__p001_loss_small,lama,lama,NaN,iopaint_lama,iopaint_lama,model_inference,restoration_eval.restoration_lama,2.0.0,cuda,...,768,True,True,lama__canonical__canonical__p001_loss_small,masked_region,54259.773041,228.083514,29.259047,NaN,True
2,lama__canonical__canonical__p001_mixed_damage,lama,lama,NaN,iopaint_lama,iopaint_lama,model_inference,restoration_eval.restoration_lama,2.0.0,cuda,...,768,True,True,lama__canonical__canonical__p001_mixed_damage,masked_region,50582.575195,214.392107,22.012804,NaN,True
3,lama__canonical__canonical__p001_scratch_thin,lama,lama,NaN,iopaint_lama,iopaint_lama,model_inference,restoration_eval.restoration_lama,2.0.0,cuda,...,768,True,True,lama__canonical__canonical__p001_scratch_thin,masked_region,49721.899544,216.654890,27.957527,NaN,True
4,lama__canonical__canonical__p001_zero_control,lama,lama,NaN,iopaint_lama,zero_control_copy,copied_zero_control,restoration_eval.restoration_lama,2.0.0,cuda,...,768,True,True,<NA>,NaN,NaN,NaN,NaN,NaN,False


In [14]:
lama_ready_cases_df = lama_cases_with_metrics_df.loc[
    lama_cases_with_metrics_df["is_lama_case"]
    & lama_cases_with_metrics_df["restoration_status_normalized"].eq("ok")
    & lama_cases_with_metrics_df["all_required_files_exist"]
    & lama_cases_with_metrics_df["content_bbox_valid"]
    & lama_cases_with_metrics_df["has_masked_metric"]
].copy()

lama_excluded_cases_df = lama_cases_with_metrics_df.loc[
    ~lama_cases_with_metrics_df.index.isin(lama_ready_cases_df.index)
].copy()

dataset_case_summary_df = (
    lama_cases_with_metrics_df.assign(
        ready_for_spatial_diagnostics=lama_cases_with_metrics_df.index.isin(
            lama_ready_cases_df.index
        )
    )
    .groupby(["dataset_name", "ready_for_spatial_diagnostics"], dropna=False)
    .size()
    .reset_index(name="case_count")
    .sort_values(["dataset_name", "ready_for_spatial_diagnostics"])
)

ready_case_checks_df = pd.DataFrame(
    [
        build_check(
            "ready_cases_nonempty",
            len(lama_ready_cases_df),
            "> 0",
            len(lama_ready_cases_df) > 0,
        ),
        build_check(
            "ready_cases_have_unique_restoration_ids",
            int(lama_ready_cases_df["restoration_case_id"].duplicated().sum()),
            0,
            not lama_ready_cases_df["restoration_case_id"].duplicated().any(),
        ),
        build_check(
            "ready_cases_have_required_datasets",
            sorted(set(lama_ready_cases_df["dataset_name"].dropna())),
            sorted(set(REQUIRED_DATASETS)),
            set(REQUIRED_DATASETS).issubset(set(lama_ready_cases_df["dataset_name"].dropna())),
        ),
    ]
)

display(dataset_case_summary_df)
display(ready_case_checks_df)

if not ready_case_checks_df["passed"].astype(bool).all():
    raise RuntimeError("LaMa ready-case validation failed.")

print("Ready LaMa cases:", len(lama_ready_cases_df))
print("Excluded LaMa cases:", len(lama_excluded_cases_df))

display(
    lama_ready_cases_df[
        [
            "restoration_case_id",
            "dataset_name",
            "case_id",
            "painting_id",
            "mask_type" if "mask_type" in lama_ready_cases_df.columns else "model_name",
            "masked_mse_improvement",
            "masked_ssim_improvement",
        ]
    ].head(10)
)

,dataset_name,ready_for_spatial_diagnostics,case_count
0,canonical,False,50
1,canonical,True,200
2,damage_size,True,35
3,mask_robustness,True,75
4,synthetic_degradation,True,50


,check,observed,expected,passed
0,ready_cases_nonempty,360,> 0,True
1,ready_cases_have_unique_restoration_ids,0,0,True
2,ready_cases_have_required_datasets,"[canonical, damage_size, mask_robustness, synt...","[canonical, damage_size]",True


Ready LaMa cases: 360
Excluded LaMa cases: 50


,restoration_case_id,dataset_name,case_id,painting_id,mask_type,masked_mse_improvement,masked_ssim_improvement
0,lama__canonical__canonical__p001_loss_large,canonical,canonical__p001_loss_large,p001,loss_large,48373.386292,NaN
1,lama__canonical__canonical__p001_loss_small,canonical,canonical__p001_loss_small,p001,loss_small,54259.773041,NaN
2,lama__canonical__canonical__p001_mixed_damage,canonical,canonical__p001_mixed_damage,p001,mixed_damage,50582.575195,NaN
3,lama__canonical__canonical__p001_scratch_thin,canonical,canonical__p001_scratch_thin,p001,scratch_thin,49721.899544,NaN
5,lama__canonical__canonical__p002_loss_large,canonical,canonical__p002_loss_large,p002,loss_large,31630.888916,NaN
6,lama__canonical__canonical__p002_loss_small,canonical,canonical__p002_loss_small,p002,loss_small,43845.346581,NaN
7,lama__canonical__canonical__p002_mixed_damage,canonical,canonical__p002_mixed_damage,p002,mixed_damage,43523.208183,NaN
8,lama__canonical__canonical__p002_scratch_thin,canonical,canonical__p002_scratch_thin,p002,scratch_thin,48152.786285,NaN
10,lama__canonical__canonical__p003_loss_large,canonical,canonical__p003_loss_large,p003,loss_large,34918.501892,NaN
11,lama__canonical__canonical__p003_loss_small,canonical,canonical__p003_loss_small,p003,loss_small,35702.919312,NaN


In [15]:
scale_case_mask_records = []

for _, row in lama_ready_cases_df.iterrows():
    mask_bool = load_mask_bool(
        Path(row["mask_path"]),
        threshold=MASK_BINARY_THRESHOLD,
    )

    mask_pixel_count = int(mask_bool.sum())
    total_pixel_count = int(mask_bool.size)

    scale_case_mask_records.append(
        {
            "restoration_case_id": row["restoration_case_id"],
            "dataset_name": row["dataset_name"],
            "case_id": row["case_id"],
            "painting_id": row["painting_id"],
            "mask_pixel_count": mask_pixel_count,
            "total_pixel_count": total_pixel_count,
            "mask_area_ratio": mask_pixel_count / total_pixel_count if total_pixel_count else np.nan,
            "has_nonzero_mask": mask_pixel_count > 0,
        }
    )

scale_case_mask_audit_df = pd.DataFrame(scale_case_mask_records)

scale_mask_summary_df = (
    scale_case_mask_audit_df
    .groupby(["dataset_name", "has_nonzero_mask"], dropna=False)
    .size()
    .reset_index(name="case_count")
    .sort_values(["dataset_name", "has_nonzero_mask"])
)

scale_mask_checks_df = pd.DataFrame(
    [
        build_check(
            "scale_cases_nonempty",
            len(scale_case_mask_audit_df),
            "> 0",
            len(scale_case_mask_audit_df) > 0,
        ),
        build_check(
            "scale_cases_have_nonzero_masks",
            int(scale_case_mask_audit_df["has_nonzero_mask"].sum()),
            "> 0",
            bool(scale_case_mask_audit_df["has_nonzero_mask"].any()),
        ),
        build_check(
            "scale_mask_ratios_valid",
            int(scale_case_mask_audit_df["mask_area_ratio"].between(0, 1).sum()),
            len(scale_case_mask_audit_df),
            bool(scale_case_mask_audit_df["mask_area_ratio"].between(0, 1).all()),
        ),
    ]
)

display(scale_mask_summary_df)
display(scale_mask_checks_df)

if not scale_mask_checks_df["passed"].astype(bool).all():
    raise RuntimeError("Scale mask coverage audit failed.")

print("Cases used for scale estimation:", len(scale_case_mask_audit_df))
print("Nonzero-mask cases:", int(scale_case_mask_audit_df["has_nonzero_mask"].sum()))
print("Zero-mask/control cases:", int((~scale_case_mask_audit_df["has_nonzero_mask"]).sum()))

,dataset_name,has_nonzero_mask,case_count
0,canonical,True,200
1,damage_size,True,35
2,mask_robustness,True,75
3,synthetic_degradation,True,50


,check,observed,expected,passed
0,scale_cases_nonempty,360,> 0,True
1,scale_cases_have_nonzero_masks,360,> 0,True
2,scale_mask_ratios_valid,360,360,True


Cases used for scale estimation: 360
Nonzero-mask cases: 360
Zero-mask/control cases: 0


In [16]:
scale_started_at = perf_counter()

visualization_scales_df = compute_global_visualization_scales(
    lama_ready_cases_df,
    clean_path_column="clean_path",
    damaged_path_column="damaged_path",
    restored_path_column="restored_path",
    mask_path_column="mask_path",
    absolute_percentile=ABSOLUTE_ERROR_PERCENTILE,
    signed_percentile=SIGNED_IMPROVEMENT_PERCENTILE,
    maximum_samples_per_case=MAX_SCALE_SAMPLES_PER_CASE,
    random_seed=SCALE_RANDOM_SEED,
    sample_region=SCALE_SAMPLE_REGION,
    mask_threshold=MASK_BINARY_THRESHOLD,
)

scale_elapsed_seconds = perf_counter() - scale_started_at

visualization_scales_df.insert(0, "notebook", NOTEBOOK_NAME)
visualization_scales_df.insert(1, "model_name", MODEL_NAME)
visualization_scales_df["spatial_schema_version"] = SPATIAL_SCHEMA_VERSION
visualization_scales_df["implementation"] = SPATIAL_IMPLEMENTATION_NAME
visualization_scales_df["run_started_at_utc"] = RUN_STARTED_AT_UTC
visualization_scales_df["created_at_utc"] = datetime.now(timezone.utc).isoformat()
visualization_scales_df["scale_elapsed_seconds"] = round(scale_elapsed_seconds, 3)
visualization_scales_df["ready_case_count"] = len(lama_ready_cases_df)
visualization_scales_df["nonzero_mask_case_count"] = int(
    scale_case_mask_audit_df["has_nonzero_mask"].sum()
)
visualization_scales_df["zero_mask_case_count"] = int(
    (~scale_case_mask_audit_df["has_nonzero_mask"]).sum()
)

display(visualization_scales_df)

print("Computed visualization scales in seconds:", round(scale_elapsed_seconds, 2))

,notebook,model_name,scale_name,cmap,vmin,vmax,percentile,sampled_value_count,sample_region,random_seed,...,sampled_case_count,mask_threshold,spatial_schema_version,implementation,run_started_at_utc,created_at_utc,scale_elapsed_seconds,ready_case_count,nonzero_mask_case_count,zero_mask_case_count
0,16_lama_difference_maps,lama,absolute_error,magma,0.0,255.0,99.5,1800000,masked,42,...,360,0,1.0.0,restoration_eval.error_maps,2026-07-28T10:53:06.469331+00:00,2026-07-28T11:09:57.501614+00:00,61.469,360,360,0
1,16_lama_difference_maps,lama,signed_improvement,coolwarm,-255.0,255.0,99.5,1800000,masked,42,...,360,0,1.0.0,restoration_eval.error_maps,2026-07-28T10:53:06.469331+00:00,2026-07-28T11:09:57.501614+00:00,61.469,360,360,0


Computed visualization scales in seconds: 61.47


In [17]:
required_scale_names = {
    "absolute_error",
    "signed_improvement",
}

required_scale_columns = {
    "notebook",
    "model_name",
    "scale_name",
    "cmap",
    "vmin",
    "vmax",
    "percentile",
    "sample_region",
    "sampled_case_count",
    "sampled_value_count",
    "ready_case_count",
    "nonzero_mask_case_count",
    "zero_mask_case_count",
    "spatial_schema_version",
    "implementation",
}

missing_scale_columns = sorted(
    required_scale_columns - set(visualization_scales_df.columns)
)

observed_scale_names = set(visualization_scales_df["scale_name"])

absolute_scale_row = visualization_scales_df.loc[
    visualization_scales_df["scale_name"].eq("absolute_error")
].iloc[0]

signed_scale_row = visualization_scales_df.loc[
    visualization_scales_df["scale_name"].eq("signed_improvement")
].iloc[0]

scale_validation_df = pd.DataFrame(
    [
        build_check(
            "scale_rows",
            len(visualization_scales_df),
            2,
            len(visualization_scales_df) == 2,
        ),
        build_check(
            "scale_names",
            sorted(observed_scale_names),
            sorted(required_scale_names),
            observed_scale_names == required_scale_names,
        ),
        build_check(
            "scale_required_columns",
            missing_scale_columns,
            [],
            not missing_scale_columns,
        ),
        build_check(
            "absolute_error_vmin_zero",
            float(absolute_scale_row["vmin"]),
            0.0,
            float(absolute_scale_row["vmin"]) == 0.0,
        ),
        build_check(
            "absolute_error_vmax_positive",
            float(absolute_scale_row["vmax"]),
            "> 0",
            float(absolute_scale_row["vmax"]) > 0,
        ),
        build_check(
            "signed_improvement_symmetric",
            round(float(signed_scale_row["vmin"]) + float(signed_scale_row["vmax"]), 8),
            0.0,
            np.isclose(float(signed_scale_row["vmin"]), -float(signed_scale_row["vmax"])),
        ),
        build_check(
            "signed_improvement_has_range",
            float(signed_scale_row["vmax"]),
            "> 0",
            float(signed_scale_row["vmax"]) > 0,
        ),
        build_check(
            "scale_sampled_case_count_matches_ready_cases",
            sorted(visualization_scales_df["sampled_case_count"].unique().tolist()),
            [len(lama_ready_cases_df)],
            set(visualization_scales_df["sampled_case_count"]) == {len(lama_ready_cases_df)},
        ),
    ]
)

display(scale_validation_df)

if not scale_validation_df["passed"].astype(bool).all():
    raise RuntimeError("Visualization scale validation failed.")

visualization_scales_df.to_csv(VISUALIZATION_SCALES_OUTPUT_PATH, index=False)

visualization_scales_readback_df = pd.read_csv(VISUALIZATION_SCALES_OUTPUT_PATH)

scale_readback_checks_df = pd.DataFrame(
    [
        build_check(
            "scale_output_exists",
            VISUALIZATION_SCALES_OUTPUT_PATH.is_file(),
            True,
            VISUALIZATION_SCALES_OUTPUT_PATH.is_file(),
        ),
        build_check(
            "scale_readback_rows",
            len(visualization_scales_readback_df),
            len(visualization_scales_df),
            len(visualization_scales_readback_df) == len(visualization_scales_df),
        ),
        build_check(
            "scale_readback_names",
            sorted(visualization_scales_readback_df["scale_name"].tolist()),
            sorted(visualization_scales_df["scale_name"].tolist()),
            sorted(visualization_scales_readback_df["scale_name"].tolist())
            == sorted(visualization_scales_df["scale_name"].tolist()),
        ),
    ]
)

display(scale_readback_checks_df)

if not scale_readback_checks_df["passed"].astype(bool).all():
    raise RuntimeError("Visualization scale readback validation failed.")

print("Saved visualization scales:", project_relative_path(VISUALIZATION_SCALES_OUTPUT_PATH))

,check,observed,expected,passed
0,scale_rows,2,2,True
1,scale_names,"[absolute_error, signed_improvement]","[absolute_error, signed_improvement]",True
2,scale_required_columns,[],[],True
3,absolute_error_vmin_zero,0.0,0.0,True
4,absolute_error_vmax_positive,255.0,> 0,True
5,signed_improvement_symmetric,0.0,0.0,True
6,signed_improvement_has_range,255.0,> 0,True
7,scale_sampled_case_count_matches_ready_cases,[360],[360],True


,check,observed,expected,passed
0,scale_output_exists,True,True,True
1,scale_readback_rows,2,2,True
2,scale_readback_names,"[absolute_error, signed_improvement]","[absolute_error, signed_improvement]",True


Saved visualization scales: data/processed/metrics/spatial_diagnostic_scales_lama.csv


In [18]:
BOUNDARY_REGION_ALIASES = {
    "boundary",
    "boundary_region",
    "mask_boundary",
    "damage_boundary",
}

SELECTED_CASES_PER_GROUP = 5
MEDIAN_CASES_PER_DATASET = 2

boundary_metrics_df = classical_metrics_normalized_df.loc[
    classical_metrics_normalized_df["evaluation_region_normalized"].isin(BOUNDARY_REGION_ALIASES)
    & classical_metrics_normalized_df["metric_status_normalized"].eq("ok")
].copy()

boundary_metric_columns = [
    "restoration_case_id",
    "mse_improvement",
    "mae_improvement",
    "psnr_improvement",
    "ssim_improvement",
]

boundary_metric_columns = [
    column_name
    for column_name in boundary_metric_columns
    if column_name in boundary_metrics_df.columns
]

if not boundary_metrics_df.empty:
    boundary_metric_lookup_df = (
        boundary_metrics_df[boundary_metric_columns]
        .drop_duplicates(subset=["restoration_case_id"], keep="first")
        .rename(
            columns={
                "mse_improvement": "boundary_mse_improvement",
                "mae_improvement": "boundary_mae_improvement",
                "psnr_improvement": "boundary_psnr_improvement",
                "ssim_improvement": "boundary_ssim_improvement",
            }
        )
    )

    lama_selection_base_df = lama_ready_cases_df.merge(
        boundary_metric_lookup_df,
        on="restoration_case_id",
        how="left",
        validate="one_to_one",
    )
else:
    lama_selection_base_df = lama_ready_cases_df.copy()
    for column_name in [
        "boundary_mse_improvement",
        "boundary_mae_improvement",
        "boundary_psnr_improvement",
        "boundary_ssim_improvement",
    ]:
        lama_selection_base_df[column_name] = np.nan

boundary_metric_audit_df = pd.DataFrame(
    [
        build_check(
            "boundary_metric_rows_available",
            len(boundary_metrics_df),
            "> 0",
            len(boundary_metrics_df) > 0,
        ),
        build_check(
            "ready_cases_with_boundary_mse",
            int(lama_selection_base_df["boundary_mse_improvement"].notna().sum()),
            f"<= {len(lama_selection_base_df)}",
            int(lama_selection_base_df["boundary_mse_improvement"].notna().sum())
            <= len(lama_selection_base_df),
        ),
    ]
)

display(boundary_metric_audit_df)

print("Selection base rows:", len(lama_selection_base_df))
print(
    "Ready cases with boundary metrics:",
    int(lama_selection_base_df["boundary_mse_improvement"].notna().sum()),
)

,check,observed,expected,passed
0,boundary_metric_rows_available,360,> 0,True
1,ready_cases_with_boundary_mse,360,<= 360,True


Selection base rows: 360
Ready cases with boundary metrics: 360


In [19]:
def append_metric_selection(
    source_df: pd.DataFrame,
    rows: list[pd.Series],
    selection_group: str,
    metric_column: str,
    ascending: bool,
    limit: int,
) -> None:
    if metric_column not in source_df.columns:
        print(f"Skipping {selection_group}: missing column {metric_column}")
        return

    valid_df = source_df.dropna(subset=[metric_column]).copy()

    if valid_df.empty:
        print(f"Skipping {selection_group}: no valid values for {metric_column}")
        return

    selected_df = valid_df.sort_values(metric_column, ascending=ascending).head(limit).copy()

    for selection_rank, (_, row) in enumerate(selected_df.iterrows(), start=1):
        selected_row = row.copy()
        selected_row["selection_group"] = selection_group
        selected_row["selection_rank"] = selection_rank
        selected_row["selection_metric"] = metric_column
        selected_row["selection_value"] = float(row[metric_column])
        selected_row["selection_reason"] = (
            f"{selection_group}: rank {selection_rank} by {metric_column}"
        )
        rows.append(selected_row)


def append_dataset_median_selection(
    source_df: pd.DataFrame,
    rows: list[pd.Series],
    selection_group: str,
    metric_column: str,
    limit_per_dataset: int,
) -> None:
    if metric_column not in source_df.columns:
        print(f"Skipping {selection_group}: missing column {metric_column}")
        return

    for dataset_name, dataset_df in source_df.groupby("dataset_name", dropna=False):
        valid_df = dataset_df.dropna(subset=[metric_column]).copy()

        if valid_df.empty:
            continue

        median_value = float(valid_df[metric_column].median())
        selected_df = (
            valid_df.assign(
                median_distance=(valid_df[metric_column] - median_value).abs()
            )
            .sort_values(["median_distance", metric_column])
            .head(limit_per_dataset)
            .copy()
        )

        for selection_rank, (_, row) in enumerate(selected_df.iterrows(), start=1):
            selected_row = row.copy()
            selected_row["selection_group"] = selection_group
            selected_row["selection_rank"] = selection_rank
            selected_row["selection_metric"] = metric_column
            selected_row["selection_value"] = float(row[metric_column])
            selected_row["selection_reason"] = (
                f"{selection_group}: {dataset_name}, closest to dataset median "
                f"{metric_column}={median_value:.6f}"
            )
            rows.append(selected_row)


def append_notebook_15_diagnostic_selection(
    source_df: pd.DataFrame,
    ready_df: pd.DataFrame,
    rows: list[pd.Series],
) -> None:
    if source_df.empty:
        print("Notebook 15 diagnostic cases not available; skipping handoff selections.")
        return

    required_columns = {"restoration_case_id", "diagnostic_reason"}

    if not required_columns.issubset(source_df.columns):
        print(
            "Notebook 15 diagnostic cases missing required columns; "
            "skipping handoff selections."
        )
        return

    handoff_df = source_df[["restoration_case_id", "diagnostic_reason"]].copy()
    handoff_df["restoration_case_id"] = normalize_string_series(
        handoff_df["restoration_case_id"]
    )
    handoff_df["diagnostic_reason"] = normalize_string_series(
        handoff_df["diagnostic_reason"]
    )

    merged_df = ready_df.merge(
        handoff_df.drop_duplicates(),
        on="restoration_case_id",
        how="inner",
    )

    if merged_df.empty:
        print("Notebook 15 diagnostic cases did not overlap ready LaMa cases.")
        return

    for selection_rank, (_, row) in enumerate(merged_df.iterrows(), start=1):
        selected_row = row.copy()
        selected_row["selection_group"] = "notebook_15_diagnostic_handoff"
        selected_row["selection_rank"] = selection_rank
        selected_row["selection_metric"] = "diagnostic_reason"
        selected_row["selection_value"] = np.nan
        selected_row["selection_reason"] = f"Notebook 15: {row['diagnostic_reason']}"
        rows.append(selected_row)

In [20]:
selection_rows = []

append_notebook_15_diagnostic_selection(
    notebook_15_diagnostic_cases_df,
    lama_selection_base_df,
    selection_rows,
)

append_metric_selection(
    lama_selection_base_df,
    selection_rows,
    selection_group="strongest_masked_mse_improvement",
    metric_column="masked_mse_improvement",
    ascending=False,
    limit=SELECTED_CASES_PER_GROUP,
)

append_metric_selection(
    lama_selection_base_df,
    selection_rows,
    selection_group="weakest_masked_mse_improvement",
    metric_column="masked_mse_improvement",
    ascending=True,
    limit=SELECTED_CASES_PER_GROUP,
)

append_dataset_median_selection(
    lama_selection_base_df,
    selection_rows,
    selection_group="dataset_median_masked_mse_improvement",
    metric_column="masked_mse_improvement",
    limit_per_dataset=MEDIAN_CASES_PER_DATASET,
)

append_metric_selection(
    lama_selection_base_df,
    selection_rows,
    selection_group="strongest_masked_ssim_improvement",
    metric_column="masked_ssim_improvement",
    ascending=False,
    limit=SELECTED_CASES_PER_GROUP,
)

append_metric_selection(
    lama_selection_base_df,
    selection_rows,
    selection_group="weakest_masked_ssim_improvement",
    metric_column="masked_ssim_improvement",
    ascending=True,
    limit=SELECTED_CASES_PER_GROUP,
)

append_metric_selection(
    lama_selection_base_df,
    selection_rows,
    selection_group="largest_boundary_improvement",
    metric_column="boundary_mse_improvement",
    ascending=False,
    limit=SELECTED_CASES_PER_GROUP,
)

append_metric_selection(
    lama_selection_base_df,
    selection_rows,
    selection_group="largest_boundary_degradation",
    metric_column="boundary_mse_improvement",
    ascending=True,
    limit=SELECTED_CASES_PER_GROUP,
)

selected_lama_cases_df = pd.DataFrame(selection_rows).reset_index(drop=True)

selected_lama_cases_df["selection_rank"] = pd.to_numeric(
    selected_lama_cases_df["selection_rank"],
    errors="coerce",
).astype("Int64")

selection_summary_df = (
    selected_lama_cases_df
    .groupby("selection_group", dropna=False)
    .agg(
        selected_rows=("restoration_case_id", "size"),
        unique_cases=("restoration_case_id", "nunique"),
        min_selection_rank=("selection_rank", "min"),
        max_selection_rank=("selection_rank", "max"),
    )
    .reset_index()
    .sort_values("selection_group")
)

display(selection_summary_df)

preview_columns = [
    "selection_group",
    "selection_rank",
    "selection_metric",
    "selection_value",
    "dataset_name",
    "case_id",
    "painting_id",
    "mask_type" if "mask_type" in selected_lama_cases_df.columns else "model_name",
    "masked_mse_improvement",
    "masked_ssim_improvement",
    "boundary_mse_improvement",
]

preview_columns = [
    column_name
    for column_name in preview_columns
    if column_name in selected_lama_cases_df.columns
]

display(selected_lama_cases_df[preview_columns].head(30))

Skipping strongest_masked_ssim_improvement: no valid values for masked_ssim_improvement
Skipping weakest_masked_ssim_improvement: no valid values for masked_ssim_improvement


,selection_group,selected_rows,unique_cases,min_selection_rank,max_selection_rank
0,dataset_median_masked_mse_improvement,8,8,1,2
1,largest_boundary_degradation,5,5,1,5
2,largest_boundary_improvement,5,5,1,5
3,notebook_15_diagnostic_handoff,12,12,1,12
4,strongest_masked_mse_improvement,5,5,1,5
5,weakest_masked_mse_improvement,5,5,1,5


,selection_group,selection_rank,selection_metric,selection_value,dataset_name,case_id,painting_id,mask_type,masked_mse_improvement,masked_ssim_improvement,boundary_mse_improvement
0,notebook_15_diagnostic_handoff,1,diagnostic_reason,NaN,canonical,canonical__p006_loss_small,p006,loss_small,49154.244480,NaN,23478.271179
1,notebook_15_diagnostic_handoff,2,diagnostic_reason,NaN,canonical,canonical__p009_loss_large,p009,loss_large,12809.674316,NaN,8445.865013
2,notebook_15_diagnostic_handoff,3,diagnostic_reason,NaN,canonical,canonical__p023_loss_small,p023,loss_small,23292.218933,NaN,11122.506851
3,notebook_15_diagnostic_handoff,4,diagnostic_reason,NaN,damage_size,damage_size__p001__loss_large__size_08pct,p001,<NA>,46074.243591,NaN,25396.999507
4,notebook_15_diagnostic_handoff,5,diagnostic_reason,NaN,damage_size,damage_size__p018__loss_large__size_02pct,p018,<NA>,19928.065674,NaN,9567.593681
5,notebook_15_diagnostic_handoff,6,diagnostic_reason,NaN,damage_size,damage_size__p026__loss_large__size_02pct,p026,<NA>,14714.078236,NaN,7380.045821
6,notebook_15_diagnostic_handoff,7,diagnostic_reason,NaN,mask_robustness,mask_robustness__p001__scratch_thin__variant_01,p001,<NA>,54879.527777,NaN,27746.510958
7,notebook_15_diagnostic_handoff,8,diagnostic_reason,NaN,mask_robustness,mask_robustness__p039__loss_large__variant_03,p039,<NA>,27179.359619,NaN,11452.502205
8,notebook_15_diagnostic_handoff,9,diagnostic_reason,NaN,mask_robustness,mask_robustness__p039__scratch_thin__variant_03,p039,<NA>,21598.311966,NaN,8265.704659
9,notebook_15_diagnostic_handoff,10,diagnostic_reason,NaN,synthetic_degradation,synthetic_degradation__p001__dirt_dust__mild,p001,<NA>,-0.547029,NaN,-3.936941


In [21]:
required_selected_columns = {
    "restoration_case_id",
    "dataset_name",
    "case_id",
    "painting_id",
    "model_name",
    "clean_path",
    "damaged_path",
    "mask_path",
    "restored_path",
    "selection_group",
    "selection_rank",
    "selection_metric",
    "selection_reason",
}

missing_selected_columns = sorted(
    required_selected_columns - set(selected_lama_cases_df.columns)
)

selected_case_validation_df = pd.DataFrame(
    [
        build_check(
            "selected_cases_nonempty",
            len(selected_lama_cases_df),
            "> 0",
            len(selected_lama_cases_df) > 0,
        ),
        build_check(
            "selected_case_required_columns",
            missing_selected_columns,
            [],
            not missing_selected_columns,
        ),
        build_check(
            "selected_cases_have_selection_group",
            int(selected_lama_cases_df["selection_group"].notna().sum()),
            len(selected_lama_cases_df),
            bool(selected_lama_cases_df["selection_group"].notna().all()),
        ),
        build_check(
            "selected_cases_have_existing_paths",
            int(
                selected_lama_cases_df[
                    ["clean_path_exists", "damaged_path_exists", "mask_path_exists", "restored_path_exists"]
                ].all(axis=1).sum()
            ),
            len(selected_lama_cases_df),
            bool(
                selected_lama_cases_df[
                    ["clean_path_exists", "damaged_path_exists", "mask_path_exists", "restored_path_exists"]
                ].all(axis=1).all()
            ),
        ),
        build_check(
            "selected_groups_available",
            selected_lama_cases_df["selection_group"].nunique(),
            ">= 4",
            selected_lama_cases_df["selection_group"].nunique() >= 4,
        ),
    ]
)

display(selected_case_validation_df)

if not selected_case_validation_df["passed"].astype(bool).all():
    raise RuntimeError("Selected LaMa case validation failed.")

print("Selected rows:", len(selected_lama_cases_df))
print("Unique selected restoration cases:", selected_lama_cases_df["restoration_case_id"].nunique())

,check,observed,expected,passed
0,selected_cases_nonempty,40,> 0,True
1,selected_case_required_columns,[],[],True
2,selected_cases_have_selection_group,40,40,True
3,selected_cases_have_existing_paths,40,40,True
4,selected_groups_available,6,>= 4,True


Selected rows: 40
Unique selected restoration cases: 37


In [22]:
required_selected_columns = {
    "restoration_case_id",
    "dataset_name",
    "case_id",
    "painting_id",
    "model_name",
    "clean_path",
    "damaged_path",
    "mask_path",
    "restored_path",
    "selection_group",
    "selection_rank",
    "selection_metric",
    "selection_reason",
}

missing_selected_columns = sorted(
    required_selected_columns - set(selected_lama_cases_df.columns)
)

selected_case_validation_df = pd.DataFrame(
    [
        build_check(
            "selected_cases_nonempty",
            len(selected_lama_cases_df),
            "> 0",
            len(selected_lama_cases_df) > 0,
        ),
        build_check(
            "selected_case_required_columns",
            missing_selected_columns,
            [],
            not missing_selected_columns,
        ),
        build_check(
            "selected_cases_have_selection_group",
            int(selected_lama_cases_df["selection_group"].notna().sum()),
            len(selected_lama_cases_df),
            bool(selected_lama_cases_df["selection_group"].notna().all()),
        ),
        build_check(
            "selected_cases_have_existing_paths",
            int(
                selected_lama_cases_df[
                    ["clean_path_exists", "damaged_path_exists", "mask_path_exists", "restored_path_exists"]
                ].all(axis=1).sum()
            ),
            len(selected_lama_cases_df),
            bool(
                selected_lama_cases_df[
                    ["clean_path_exists", "damaged_path_exists", "mask_path_exists", "restored_path_exists"]
                ].all(axis=1).all()
            ),
        ),
        build_check(
            "selected_groups_available",
            selected_lama_cases_df["selection_group"].nunique(),
            ">= 4",
            selected_lama_cases_df["selection_group"].nunique() >= 4,
        ),
    ]
)

display(selected_case_validation_df)

if not selected_case_validation_df["passed"].astype(bool).all():
    raise RuntimeError("Selected LaMa case validation failed.")

print("Selected rows:", len(selected_lama_cases_df))
print("Unique selected restoration cases:", selected_lama_cases_df["restoration_case_id"].nunique())

,check,observed,expected,passed
0,selected_cases_nonempty,40,> 0,True
1,selected_case_required_columns,[],[],True
2,selected_cases_have_selection_group,40,40,True
3,selected_cases_have_existing_paths,40,40,True
4,selected_groups_available,6,>= 4,True


Selected rows: 40
Unique selected restoration cases: 37


In [23]:
selected_figure_cases_df = (
    selected_lama_cases_df
    .sort_values(["selection_group", "selection_rank", "restoration_case_id"])
    .drop_duplicates(subset=["selection_group", "restoration_case_id"], keep="first")
    .reset_index(drop=True)
    .copy()
)

selected_figure_case_checks_df = pd.DataFrame(
    [
        build_check(
            "selected_figure_cases_nonempty",
            len(selected_figure_cases_df),
            "> 0",
            len(selected_figure_cases_df) > 0,
        ),
        build_check(
            "selected_figure_rows_not_more_than_selected_rows",
            len(selected_figure_cases_df),
            f"<= {len(selected_lama_cases_df)}",
            len(selected_figure_cases_df) <= len(selected_lama_cases_df),
        ),
        build_check(
            "selected_group_case_pairs_unique",
            int(
                selected_figure_cases_df[
                    ["selection_group", "restoration_case_id"]
                ].duplicated().sum()
            ),
            0,
            not selected_figure_cases_df[
                ["selection_group", "restoration_case_id"]
            ].duplicated().any(),
        ),
    ]
)

display(selected_figure_case_checks_df)

if not selected_figure_case_checks_df["passed"].astype(bool).all():
    raise RuntimeError("Selected figure case preparation failed.")

print("Selected rows from Batch 4:", len(selected_lama_cases_df))
print("Selected figure rows after group/case de-duplication:", len(selected_figure_cases_df))

display(
    selected_figure_cases_df[
        [
            "selection_group",
            "selection_rank",
            "restoration_case_id",
            "dataset_name",
            "case_id",
            "painting_id",
            "selection_metric",
            "selection_value",
        ]
    ].head(20)
)

,check,observed,expected,passed
0,selected_figure_cases_nonempty,40,> 0,True
1,selected_figure_rows_not_more_than_selected_rows,40,<= 40,True
2,selected_group_case_pairs_unique,0,0,True


Selected rows from Batch 4: 40
Selected figure rows after group/case de-duplication: 40


,selection_group,selection_rank,restoration_case_id,dataset_name,case_id,painting_id,selection_metric,selection_value
0,dataset_median_masked_mse_improvement,1,lama__canonical__canonical__p007_loss_large,canonical,canonical__p007_loss_large,p007,masked_mse_improvement,27140.822754
1,dataset_median_masked_mse_improvement,1,lama__damage_size__damage_size__p018__loss_lar...,damage_size,damage_size__p018__loss_large__size_08pct,p018,masked_mse_improvement,23642.714783
2,dataset_median_masked_mse_improvement,1,lama__mask_robustness__mask_robustness__p026__...,mask_robustness,mask_robustness__p026__scratch_thin__variant_05,p026,masked_mse_improvement,28711.128632
3,dataset_median_masked_mse_improvement,1,lama__synthetic_degradation__synthetic_degrada...,synthetic_degradation,synthetic_degradation__p026__water_stain_dirt_...,p026,masked_mse_improvement,-15.249096
4,dataset_median_masked_mse_improvement,2,lama__canonical__canonical__p012_scratch_thin,canonical,canonical__p012_scratch_thin,p012,masked_mse_improvement,27181.320129
5,dataset_median_masked_mse_improvement,2,lama__damage_size__damage_size__p018__loss_lar...,damage_size,damage_size__p018__loss_large__size_10pct,p018,masked_mse_improvement,23803.383545
6,dataset_median_masked_mse_improvement,2,lama__mask_robustness__mask_robustness__p043__...,mask_robustness,mask_robustness__p043__loss_large__variant_04,p043,masked_mse_improvement,28982.953552
7,dataset_median_masked_mse_improvement,2,lama__synthetic_degradation__synthetic_degrada...,synthetic_degradation,synthetic_degradation__p001__water_stain__severe,p001,masked_mse_improvement,-9.966362
8,largest_boundary_degradation,1,lama__synthetic_degradation__synthetic_degrada...,synthetic_degradation,synthetic_degradation__p039__dirt_dust__severe,p039,boundary_mse_improvement,-24.388120
9,largest_boundary_degradation,2,lama__synthetic_degradation__synthetic_degrada...,synthetic_degradation,synthetic_degradation__p039__dirt_dust__moderate,p039,boundary_mse_improvement,-22.047948


In [25]:
active_visualization_scales_df = (
    visualization_scales_df.copy()
    if "visualization_scales_df" in globals()
    else pd.read_csv(VISUALIZATION_SCALES_OUTPUT_PATH)
)

scale_lookup_df = active_visualization_scales_df.set_index("scale_name")

ERROR_VMIN = float(scale_lookup_df.loc["absolute_error", "vmin"])
ERROR_VMAX = float(scale_lookup_df.loc["absolute_error", "vmax"])
IMPROVEMENT_VMIN = float(scale_lookup_df.loc["signed_improvement", "vmin"])
IMPROVEMENT_VMAX = float(scale_lookup_df.loc["signed_improvement", "vmax"])

print("ERROR_VMIN:", ERROR_VMIN)
print("ERROR_VMAX:", ERROR_VMAX)
print("IMPROVEMENT_VMIN:", IMPROVEMENT_VMIN)
print("IMPROVEMENT_VMAX:", IMPROVEMENT_VMAX)

ERROR_VMIN: 0.0
ERROR_VMAX: 255.0
IMPROVEMENT_VMIN: -255.0
IMPROVEMENT_VMAX: 255.0


In [26]:
selected_generation_started_at = perf_counter()

selected_figure_manifest_df = generate_error_map_figures_for_cases(
    selected_figure_cases_df,
    output_dir=SELECTED_CASE_FIGURES_DIR,
    selection_group_column="selection_group",
    error_vmin=ERROR_VMIN,
    error_vmax=ERROR_VMAX,
    improvement_vmin=IMPROVEMENT_VMIN,
    improvement_vmax=IMPROVEMENT_VMAX,
    boundary_width_pixels=BOUNDARY_WIDTH_PIXELS,
    boundary_mode=BOUNDARY_MODE,
    mask_threshold=MASK_BINARY_THRESHOLD,
    show=False,
    dpi=FIGURE_DPI,
    progress_every=10,
)

selected_generation_elapsed_seconds = perf_counter() - selected_generation_started_at

selected_manifest_metadata_df = selected_figure_cases_df[
    [
        "restoration_case_id",
        "selection_group",
        "selection_rank",
        "selection_reason",
    ]
].copy()

selected_figure_manifest_df = selected_figure_manifest_df.merge(
    selected_manifest_metadata_df,
    on=["restoration_case_id", "selection_group"],
    how="left",
    validate="one_to_one",
)

selected_figure_manifest_df.insert(0, "notebook", NOTEBOOK_NAME)
selected_figure_manifest_df.insert(1, "spatial_schema_version", SPATIAL_SCHEMA_VERSION)
selected_figure_manifest_df.insert(2, "implementation", SPATIAL_IMPLEMENTATION_NAME)
selected_figure_manifest_df["run_started_at_utc"] = RUN_STARTED_AT_UTC
selected_figure_manifest_df["created_at_utc"] = datetime.now(timezone.utc).isoformat()
selected_figure_manifest_df["figure_set"] = "selected"
selected_figure_manifest_df["figure_path_project_relative"] = selected_figure_manifest_df[
    "figure_path"
].map(project_relative_path)
selected_figure_manifest_df["generation_elapsed_seconds_total"] = round(
    selected_generation_elapsed_seconds,
    3,
)

display(selected_figure_manifest_df.head())

print(
    "Generated selected figures in seconds:",
    round(selected_generation_elapsed_seconds, 2),
)

Starting error-map generation for 40 cases...
Output directory: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\selected_cases
Processed 1/40 cases...
Processed 10/40 cases...
Processed 20/40 cases...
Processed 30/40 cases...
Processed 40/40 cases...
Error-map generation finished.


,notebook,spatial_schema_version,implementation,restoration_case_id,case_id,source_case_id,dataset_name,painting_id,category,title,...,positive_improvement_percentage_masked,status,issue,selection_rank,selection_reason,run_started_at_utc,created_at_utc,figure_set,figure_path_project_relative,generation_elapsed_seconds_total
0,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,lama__canonical__canonical__p007_loss_large,canonical__p007_loss_large,canonical__p007_loss_large,canonical,p007,portrait_figure,Princesse de Broglie,...,95.222824,ok,,1,dataset_median_masked_mse_improvement: canonic...,2026-07-28T10:53:06.469331+00:00,2026-07-28T11:28:16.810362+00:00,selected,outputs/16_lama_difference_maps/figures/select...,242.446
1,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,lama__damage_size__damage_size__p018__loss_lar...,damage_size__p018__loss_large__size_08pct,damage_size__p018__loss_large__size_08pct,damage_size,p018,landscape_natural,Classical Landscape with Figures,...,99.977556,ok,,1,dataset_median_masked_mse_improvement: damage_...,2026-07-28T10:53:06.469331+00:00,2026-07-28T11:28:16.810362+00:00,selected,outputs/16_lama_difference_maps/figures/select...,242.446
2,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,lama__mask_robustness__mask_robustness__p026__...,mask_robustness__p026__scratch_thin__variant_05,mask_robustness__p026__scratch_thin__variant_05,mask_robustness,p026,architecture_structured,View of a Village along a River,...,100.000000,ok,,1,dataset_median_masked_mse_improvement: mask_ro...,2026-07-28T10:53:06.469331+00:00,2026-07-28T11:28:16.810362+00:00,selected,outputs/16_lama_difference_maps/figures/select...,242.446
3,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,lama__synthetic_degradation__synthetic_degrada...,synthetic_degradation__p026__water_stain_dirt_...,synthetic_degradation__p026__water_stain_dirt_...,synthetic_degradation,p026,architecture_structured,View of a Village along a River,...,1.657366,ok,,1,dataset_median_masked_mse_improvement: synthet...,2026-07-28T10:53:06.469331+00:00,2026-07-28T11:28:16.810362+00:00,selected,outputs/16_lama_difference_maps/figures/select...,242.446
4,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,lama__canonical__canonical__p012_scratch_thin,canonical__p012_scratch_thin,canonical__p012_scratch_thin,canonical,p012,landscape_natural,An Extensive Wooded Landscape,...,99.686815,ok,,2,dataset_median_masked_mse_improvement: canonic...,2026-07-28T10:53:06.469331+00:00,2026-07-28T11:28:16.810362+00:00,selected,outputs/16_lama_difference_maps/figures/select...,242.446


Generated selected figures in seconds: 242.45


In [27]:
selected_manifest_validation_df = validate_error_map_manifest(
    selected_figure_manifest_df,
    expected_rows=len(selected_figure_cases_df),
    require_unique_case_ids=False,
    require_nonempty_figures=True,
)

display(selected_manifest_validation_df)

if not selected_manifest_validation_df["passed"].astype(bool).all():
    display(
        selected_figure_manifest_df.loc[
            selected_figure_manifest_df["status"].ne("ok"),
            [
                "restoration_case_id",
                "selection_group",
                "figure_path",
                "status",
                "issue",
            ],
        ].head(20)
    )
    raise RuntimeError("Selected figure manifest validation failed.")

selected_figure_manifest_df.to_csv(SELECTED_FIGURE_MANIFEST_PATH, index=False)

selected_figure_manifest_readback_df = pd.read_csv(SELECTED_FIGURE_MANIFEST_PATH)

selected_manifest_readback_checks_df = pd.DataFrame(
    [
        build_check(
            "selected_manifest_output_exists",
            SELECTED_FIGURE_MANIFEST_PATH.is_file(),
            True,
            SELECTED_FIGURE_MANIFEST_PATH.is_file(),
        ),
        build_check(
            "selected_manifest_readback_rows",
            len(selected_figure_manifest_readback_df),
            len(selected_figure_manifest_df),
            len(selected_figure_manifest_readback_df) == len(selected_figure_manifest_df),
        ),
        build_check(
            "selected_manifest_readback_status_ok",
            int(selected_figure_manifest_readback_df["status"].eq("ok").sum()),
            len(selected_figure_manifest_readback_df),
            bool(selected_figure_manifest_readback_df["status"].eq("ok").all()),
        ),
    ]
)

display(selected_manifest_readback_checks_df)

if not selected_manifest_readback_checks_df["passed"].astype(bool).all():
    raise RuntimeError("Selected figure manifest readback validation failed.")

print("Saved selected figure manifest:", project_relative_path(SELECTED_FIGURE_MANIFEST_PATH))
print("Selected figures generated:", len(selected_figure_manifest_df))

,check,passed,detail
0,required_columns,True,All required columns present.
1,row_count,True,"Expected 40, found 40."
2,status_ok,True,Rows with non-ok status: 0.
3,figures_exist,True,Missing figure files: 0.
4,figures_nonempty,True,Empty or missing figure files: 0.


,check,observed,expected,passed
0,selected_manifest_output_exists,True,True,True
1,selected_manifest_readback_rows,40,40,True
2,selected_manifest_readback_status_ok,40,40,True


Saved selected figure manifest: outputs/16_lama_difference_maps/lama_difference_map_manifest_selected.csv
Selected figures generated: 40


In [28]:
def extract_content_bbox_for_spatial_summary(row: pd.Series) -> tuple[int, int, int, int] | None:
    candidate_column_sets = [
        ("content_x_min", "content_y_min", "content_x_max", "content_y_max"),
        ("content_bbox_x_min", "content_bbox_y_min", "content_bbox_x_max", "content_bbox_y_max"),
        ("content_bbox_left", "content_bbox_top", "content_bbox_right", "content_bbox_bottom"),
    ]

    for column_names in candidate_column_sets:
        if not all(column_name in row.index for column_name in column_names):
            continue

        values = [row[column_name] for column_name in column_names]

        if any(pd.isna(value) for value in values):
            continue

        x_min, y_min, x_max, y_max = [int(round(float(value))) for value in values]

        if x_max > x_min and y_max > y_min:
            return x_min, y_min, x_max, y_max

    return None


def build_spatial_case_metadata(row: pd.Series) -> dict:
    metadata_columns = [
        "restoration_case_id",
        "dataset_name",
        "case_id",
        "painting_id",
        "source_case_id",
        "model_name",
        "mask_type",
        "damage_type",
        "damage_variant",
        "clean_path",
        "damaged_path",
        "mask_path",
        "restored_path",
        "masked_mse_improvement",
        "masked_mae_improvement",
        "masked_psnr_improvement",
        "masked_ssim_improvement",
        "boundary_mse_improvement",
        "boundary_mae_improvement",
        "boundary_psnr_improvement",
        "boundary_ssim_improvement",
    ]

    return {
        column_name: row.get(column_name, np.nan)
        for column_name in metadata_columns
        if column_name in row.index
    }

In [29]:
spatial_diagnostic_records = []
spatial_diagnostics_started_at = perf_counter()

for case_number, (_, row) in enumerate(lama_ready_cases_df.iterrows(), start=1):
    status = "ok"
    issue = ""

    try:
        content_bbox = extract_content_bbox_for_spatial_summary(row)

        summary = compute_error_map_summary(
            clean_path=Path(row["clean_path"]),
            damaged_path=Path(row["damaged_path"]),
            restored_path=Path(row["restored_path"]),
            mask_path=Path(row["mask_path"]),
            content_bbox=content_bbox,
            boundary_width_pixels=BOUNDARY_WIDTH_PIXELS,
            boundary_mode=BOUNDARY_MODE,
            mask_threshold=MASK_BINARY_THRESHOLD,
        )

        record = {
            "notebook": NOTEBOOK_NAME,
            "spatial_schema_version": SPATIAL_SCHEMA_VERSION,
            "implementation": SPATIAL_IMPLEMENTATION_NAME,
            "model_name": MODEL_NAME,
            "run_started_at_utc": RUN_STARTED_AT_UTC,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "diagnostic_status": status,
            "diagnostic_issue": issue,
            "content_bbox_used": content_bbox is not None,
            "content_bbox_x_min": content_bbox[0] if content_bbox is not None else np.nan,
            "content_bbox_y_min": content_bbox[1] if content_bbox is not None else np.nan,
            "content_bbox_x_max": content_bbox[2] if content_bbox is not None else np.nan,
            "content_bbox_y_max": content_bbox[3] if content_bbox is not None else np.nan,
            **build_spatial_case_metadata(row),
            **summary,
        }

    except Exception as exc:
        status = "error"
        issue = f"{type(exc).__name__}: {exc}"

        record = {
            "notebook": NOTEBOOK_NAME,
            "spatial_schema_version": SPATIAL_SCHEMA_VERSION,
            "implementation": SPATIAL_IMPLEMENTATION_NAME,
            "model_name": MODEL_NAME,
            "run_started_at_utc": RUN_STARTED_AT_UTC,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "diagnostic_status": status,
            "diagnostic_issue": issue,
            **build_spatial_case_metadata(row),
        }

    spatial_diagnostic_records.append(record)

    if case_number == 1 or case_number % 25 == 0 or case_number == len(lama_ready_cases_df):
        print(f"Computed spatial diagnostics for {case_number}/{len(lama_ready_cases_df)} cases...")

spatial_diagnostics_elapsed_seconds = perf_counter() - spatial_diagnostics_started_at

spatial_diagnostics_df = pd.DataFrame(spatial_diagnostic_records)
spatial_diagnostics_df["diagnostic_elapsed_seconds_total"] = round(
    spatial_diagnostics_elapsed_seconds,
    3,
)

display(spatial_diagnostics_df.head())

print(
    "Computed spatial diagnostics in seconds:",
    round(spatial_diagnostics_elapsed_seconds, 2),
)

Computed spatial diagnostics for 1/360 cases...
Computed spatial diagnostics for 25/360 cases...
Computed spatial diagnostics for 50/360 cases...
Computed spatial diagnostics for 75/360 cases...
Computed spatial diagnostics for 100/360 cases...
Computed spatial diagnostics for 125/360 cases...
Computed spatial diagnostics for 150/360 cases...
Computed spatial diagnostics for 175/360 cases...
Computed spatial diagnostics for 200/360 cases...
Computed spatial diagnostics for 225/360 cases...
Computed spatial diagnostics for 250/360 cases...
Computed spatial diagnostics for 275/360 cases...
Computed spatial diagnostics for 300/360 cases...
Computed spatial diagnostics for 325/360 cases...
Computed spatial diagnostics for 350/360 cases...
Computed spatial diagnostics for 360/360 cases...


,notebook,spatial_schema_version,implementation,model_name,run_started_at_utc,created_at_utc,diagnostic_status,diagnostic_issue,content_bbox_used,content_bbox_x_min,...,restored_error_outside_mask_std,damaged_error_mean_masked,restored_error_mean_masked,improvement_mean_masked,negative_improvement_pixels_masked,positive_improvement_pixels_masked,zero_improvement_pixels_masked,negative_improvement_percentage_masked,positive_improvement_percentage_masked,diagnostic_elapsed_seconds_total
0,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,lama,2026-07-28T10:53:06.469331+00:00,2026-07-28T11:31:27.413095+00:00,ok,,True,52,...,0.0,220.027664,10.892377,209.135269,54,68963,1,0.078240,99.920311,83.945
1,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,lama,2026-07-28T10:53:06.469331+00:00,2026-07-28T11:31:27.641447+00:00,ok,,True,52,...,0.0,232.473816,4.390317,228.083511,0,18239,0,0.000000,100.000000,83.945
2,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,lama,2026-07-28T10:53:06.469331+00:00,2026-07-28T11:31:27.863709+00:00,ok,,True,52,...,0.0,223.858749,9.466658,214.392105,0,45733,0,0.000000,100.000000,83.945
3,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,lama,2026-07-28T10:53:06.469331+00:00,2026-07-28T11:31:28.105628+00:00,ok,,True,52,...,0.0,221.437164,4.782275,216.654892,0,12058,0,0.000000,100.000000,83.945
4,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,lama,2026-07-28T10:53:06.469331+00:00,2026-07-28T11:31:28.323946+00:00,ok,,True,159,...,0.0,180.486572,29.318581,151.167999,5182,32243,14,13.841182,86.121424,83.945


Computed spatial diagnostics in seconds: 83.95


In [30]:
required_spatial_diagnostic_columns = {
    "notebook",
    "spatial_schema_version",
    "implementation",
    "model_name",
    "restoration_case_id",
    "dataset_name",
    "case_id",
    "painting_id",
    "diagnostic_status",
    "diagnostic_issue",
    "image_height",
    "image_width",
    "masked_pixel_count",
    "boundary_pixel_count",
    "damaged_error_mean_full",
    "restored_error_mean_full",
    "improvement_mean_full",
    "improvement_mean_masked",
    "positive_improvement_percentage_masked",
    "negative_improvement_percentage_masked",
}

missing_spatial_diagnostic_columns = sorted(
    required_spatial_diagnostic_columns - set(spatial_diagnostics_df.columns)
)

spatial_diagnostic_validation_df = pd.DataFrame(
    [
        build_check(
            "spatial_diagnostic_rows",
            len(spatial_diagnostics_df),
            len(lama_ready_cases_df),
            len(spatial_diagnostics_df) == len(lama_ready_cases_df),
        ),
        build_check(
            "spatial_diagnostic_required_columns",
            missing_spatial_diagnostic_columns,
            [],
            not missing_spatial_diagnostic_columns,
        ),
        build_check(
            "spatial_diagnostic_status_ok",
            int(spatial_diagnostics_df["diagnostic_status"].eq("ok").sum()),
            len(spatial_diagnostics_df),
            bool(spatial_diagnostics_df["diagnostic_status"].eq("ok").all()),
        ),
        build_check(
            "spatial_diagnostic_unique_restoration_ids",
            int(spatial_diagnostics_df["restoration_case_id"].duplicated().sum()),
            0,
            not spatial_diagnostics_df["restoration_case_id"].duplicated().any(),
        ),
        build_check(
            "spatial_diagnostic_nonzero_masked_cases",
            int(spatial_diagnostics_df["masked_pixel_count"].fillna(0).gt(0).sum()),
            "> 0",
            bool(spatial_diagnostics_df["masked_pixel_count"].fillna(0).gt(0).any()),
        ),
    ]
)

display(spatial_diagnostic_validation_df)

if not spatial_diagnostic_validation_df["passed"].astype(bool).all():
    display(
        spatial_diagnostics_df.loc[
            spatial_diagnostics_df["diagnostic_status"].ne("ok"),
            [
                "restoration_case_id",
                "dataset_name",
                "case_id",
                "painting_id",
                "diagnostic_status",
                "diagnostic_issue",
            ],
        ].head(30)
    )
    raise RuntimeError("Spatial diagnostics validation failed.")

spatial_diagnostics_df.to_csv(SPATIAL_DIAGNOSTICS_OUTPUT_PATH, index=False)

spatial_diagnostics_readback_df = pd.read_csv(SPATIAL_DIAGNOSTICS_OUTPUT_PATH)

spatial_diagnostics_readback_checks_df = pd.DataFrame(
    [
        build_check(
            "spatial_diagnostics_output_exists",
            SPATIAL_DIAGNOSTICS_OUTPUT_PATH.is_file(),
            True,
            SPATIAL_DIAGNOSTICS_OUTPUT_PATH.is_file(),
        ),
        build_check(
            "spatial_diagnostics_readback_rows",
            len(spatial_diagnostics_readback_df),
            len(spatial_diagnostics_df),
            len(spatial_diagnostics_readback_df) == len(spatial_diagnostics_df),
        ),
        build_check(
            "spatial_diagnostics_readback_status_ok",
            int(spatial_diagnostics_readback_df["diagnostic_status"].eq("ok").sum()),
            len(spatial_diagnostics_readback_df),
            bool(spatial_diagnostics_readback_df["diagnostic_status"].eq("ok").all()),
        ),
    ]
)

display(spatial_diagnostics_readback_checks_df)

if not spatial_diagnostics_readback_checks_df["passed"].astype(bool).all():
    raise RuntimeError("Spatial diagnostics readback validation failed.")

print("Saved spatial diagnostics:", project_relative_path(SPATIAL_DIAGNOSTICS_OUTPUT_PATH))
print("Spatial diagnostic rows:", len(spatial_diagnostics_df))

,check,observed,expected,passed
0,spatial_diagnostic_rows,360,360,True
1,spatial_diagnostic_required_columns,[],[],True
2,spatial_diagnostic_status_ok,360,360,True
3,spatial_diagnostic_unique_restoration_ids,0,0,True
4,spatial_diagnostic_nonzero_masked_cases,360,> 0,True


,check,observed,expected,passed
0,spatial_diagnostics_output_exists,True,True,True
1,spatial_diagnostics_readback_rows,360,360,True
2,spatial_diagnostics_readback_status_ok,360,360,True


Saved spatial diagnostics: data/processed/metrics/spatial_diagnostics_lama.csv
Spatial diagnostic rows: 360


In [31]:
active_spatial_diagnostics_df = (
    spatial_diagnostics_df.copy()
    if "spatial_diagnostics_df" in globals()
    else pd.read_csv(SPATIAL_DIAGNOSTICS_OUTPUT_PATH)
)

active_selected_manifest_df = (
    selected_figure_manifest_df.copy()
    if "selected_figure_manifest_df" in globals()
    else pd.read_csv(SELECTED_FIGURE_MANIFEST_PATH)
)

active_manifest_checks_df = pd.DataFrame(
    [
        build_check(
            "spatial_diagnostics_available",
            len(active_spatial_diagnostics_df),
            len(lama_ready_cases_df),
            len(active_spatial_diagnostics_df) == len(lama_ready_cases_df),
        ),
        build_check(
            "selected_manifest_available",
            len(active_selected_manifest_df),
            "> 0",
            len(active_selected_manifest_df) > 0,
        ),
        build_check(
            "selected_manifest_status_ok",
            int(active_selected_manifest_df["status"].eq("ok").sum()),
            len(active_selected_manifest_df),
            bool(active_selected_manifest_df["status"].eq("ok").all()),
        ),
    ]
)

display(active_manifest_checks_df)

if not active_manifest_checks_df["passed"].astype(bool).all():
    raise RuntimeError("Active spatial manifest inputs are not ready.")

,check,observed,expected,passed
0,spatial_diagnostics_available,360,360,True
1,selected_manifest_available,40,> 0,True
2,selected_manifest_status_ok,40,40,True


In [32]:
if GENERATE_ALL_CASE_FIGURES:
    all_case_figure_cases_df = lama_ready_cases_df.copy()

    if not INCLUDE_ZERO_CONTROL_FIGURES and "masked_pixel_count" in active_spatial_diagnostics_df.columns:
        nonzero_mask_ids = set(
            active_spatial_diagnostics_df.loc[
                active_spatial_diagnostics_df["masked_pixel_count"].fillna(0).gt(0),
                "restoration_case_id",
            ]
        )

        all_case_figure_cases_df = all_case_figure_cases_df.loc[
            all_case_figure_cases_df["restoration_case_id"].isin(nonzero_mask_ids)
        ].copy()

    all_generation_started_at = perf_counter()

    all_figure_manifest_df = generate_error_map_figures_for_cases(
        all_case_figure_cases_df,
        output_dir=ALL_CASE_FIGURES_DIR,
        error_vmin=ERROR_VMIN,
        error_vmax=ERROR_VMAX,
        improvement_vmin=IMPROVEMENT_VMIN,
        improvement_vmax=IMPROVEMENT_VMAX,
        boundary_width_pixels=BOUNDARY_WIDTH_PIXELS,
        boundary_mode=BOUNDARY_MODE,
        mask_threshold=MASK_BINARY_THRESHOLD,
        show=False,
        dpi=FIGURE_DPI,
        progress_every=25,
    )

    all_generation_elapsed_seconds = perf_counter() - all_generation_started_at

    all_figure_manifest_df.insert(0, "notebook", NOTEBOOK_NAME)
    all_figure_manifest_df.insert(1, "spatial_schema_version", SPATIAL_SCHEMA_VERSION)
    all_figure_manifest_df.insert(2, "implementation", SPATIAL_IMPLEMENTATION_NAME)
    all_figure_manifest_df["run_started_at_utc"] = RUN_STARTED_AT_UTC
    all_figure_manifest_df["created_at_utc"] = datetime.now(timezone.utc).isoformat()
    all_figure_manifest_df["figure_set"] = "all_cases"
    all_figure_manifest_df["figure_path_project_relative"] = all_figure_manifest_df[
        "figure_path"
    ].map(project_relative_path)
    all_figure_manifest_df["generation_elapsed_seconds_total"] = round(
        all_generation_elapsed_seconds,
        3,
    )

    all_manifest_validation_df = validate_error_map_manifest(
        all_figure_manifest_df,
        expected_rows=len(all_case_figure_cases_df),
        require_unique_case_ids=True,
        require_nonempty_figures=True,
    )

    display(all_manifest_validation_df)

    if not all_manifest_validation_df["passed"].astype(bool).all():
        raise RuntimeError("All-case figure manifest validation failed.")

    all_figure_manifest_df.to_csv(ALL_FIGURE_MANIFEST_PATH, index=False)

    print("Saved all-case figure manifest:", project_relative_path(ALL_FIGURE_MANIFEST_PATH))
    print("All-case figures generated:", len(all_figure_manifest_df))

else:
    all_figure_manifest_df = pd.DataFrame()
    print("Skipped all-case figure generation because GENERATE_ALL_CASE_FIGURES=False.")

Starting error-map generation for 360 cases...
Output directory: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\16_lama_difference_maps\figures\all_cases
Processed 1/360 cases...
Processed 25/360 cases...
Processed 50/360 cases...
Processed 75/360 cases...
Processed 100/360 cases...
Processed 125/360 cases...
Processed 150/360 cases...
Processed 175/360 cases...
Processed 200/360 cases...
Processed 225/360 cases...
Processed 250/360 cases...
Processed 275/360 cases...
Processed 300/360 cases...
Processed 325/360 cases...
Processed 350/360 cases...
Processed 360/360 cases...
Error-map generation finished.


,check,passed,detail
0,required_columns,True,All required columns present.
1,row_count,True,"Expected 360, found 360."
2,unique_case_ids,True,Duplicate case IDs: 0.
3,status_ok,True,Rows with non-ok status: 0.
4,figures_exist,True,Missing figure files: 0.
5,figures_nonempty,True,Empty or missing figure files: 0.


Saved all-case figure manifest: outputs/16_lama_difference_maps/lama_difference_map_manifest_all.csv
All-case figures generated: 360


In [33]:
selected_asset_summary_df = (
    active_selected_manifest_df
    .groupby("restoration_case_id", dropna=False)
    .agg(
        selected_figure_count=("figure_path", "size"),
        selected_selection_groups=(
            "selection_group",
            lambda values: " | ".join(sorted(set(map(str, values)))),
        ),
        selected_figure_paths_project_relative=(
            "figure_path_project_relative",
            lambda values: " | ".join(map(str, values)),
        ),
    )
    .reset_index()
)

if not all_figure_manifest_df.empty:
    all_asset_summary_df = all_figure_manifest_df[
        [
            "restoration_case_id",
            "figure_path",
            "figure_path_project_relative",
            "status",
        ]
    ].rename(
        columns={
            "figure_path": "all_case_figure_path",
            "figure_path_project_relative": "all_case_figure_path_project_relative",
            "status": "all_case_figure_status",
        }
    )
else:
    all_asset_summary_df = pd.DataFrame(
        columns=[
            "restoration_case_id",
            "all_case_figure_path",
            "all_case_figure_path_project_relative",
            "all_case_figure_status",
        ]
    )

case_asset_index_df = lama_ready_cases_df.merge(
    active_spatial_diagnostics_df[
        [
            "restoration_case_id",
            "diagnostic_status",
            "masked_pixel_count",
            "boundary_pixel_count",
            "improvement_mean_masked",
            "positive_improvement_percentage_masked",
            "negative_improvement_percentage_masked",
        ]
    ],
    on="restoration_case_id",
    how="left",
    validate="one_to_one",
)

case_asset_index_df = case_asset_index_df.merge(
    selected_asset_summary_df,
    on="restoration_case_id",
    how="left",
    validate="one_to_one",
)

case_asset_index_df = case_asset_index_df.merge(
    all_asset_summary_df,
    on="restoration_case_id",
    how="left",
    validate="one_to_one",
)

case_asset_index_df["notebook"] = NOTEBOOK_NAME
case_asset_index_df["spatial_schema_version"] = SPATIAL_SCHEMA_VERSION
case_asset_index_df["implementation"] = SPATIAL_IMPLEMENTATION_NAME
case_asset_index_df["created_at_utc"] = datetime.now(timezone.utc).isoformat()

case_asset_index_df["selected_figure_count"] = (
    case_asset_index_df["selected_figure_count"].fillna(0).astype(int)
)
case_asset_index_df["has_selected_figure"] = case_asset_index_df["selected_figure_count"].gt(0)
case_asset_index_df["has_all_case_figure"] = case_asset_index_df[
    "all_case_figure_path"
].notna()

for path_column in ["clean_path", "damaged_path", "mask_path", "restored_path"]:
    case_asset_index_df[f"{path_column}_project_relative"] = case_asset_index_df[
        path_column
    ].map(project_relative_path)

display(case_asset_index_df.head())

,restoration_case_id,model_name,iopaint_model_name,iopaint_package_version,lama_model_version,restoration_method,inference_mode,restoration_generator_name,restoration_generator_version,execution_device,...,selected_figure_paths_project_relative,all_case_figure_path,all_case_figure_path_project_relative,all_case_figure_status,notebook,spatial_schema_version,implementation,created_at_utc,has_selected_figure,has_all_case_figure
0,lama__canonical__canonical__p001_loss_large,lama,lama,NaN,iopaint_lama,iopaint_lama,model_inference,restoration_eval.restoration_lama,2.0.0,cuda,...,NaN,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/16_lama_difference_maps/figures/all_ca...,ok,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,2026-07-28T12:19:32.929592+00:00,False,True
1,lama__canonical__canonical__p001_loss_small,lama,lama,NaN,iopaint_lama,iopaint_lama,model_inference,restoration_eval.restoration_lama,2.0.0,cuda,...,outputs/16_lama_difference_maps/figures/select...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/16_lama_difference_maps/figures/all_ca...,ok,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,2026-07-28T12:19:32.929592+00:00,True,True
2,lama__canonical__canonical__p001_mixed_damage,lama,lama,NaN,iopaint_lama,iopaint_lama,model_inference,restoration_eval.restoration_lama,2.0.0,cuda,...,NaN,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/16_lama_difference_maps/figures/all_ca...,ok,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,2026-07-28T12:19:32.929592+00:00,False,True
3,lama__canonical__canonical__p001_scratch_thin,lama,lama,NaN,iopaint_lama,iopaint_lama,model_inference,restoration_eval.restoration_lama,2.0.0,cuda,...,NaN,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/16_lama_difference_maps/figures/all_ca...,ok,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,2026-07-28T12:19:32.929592+00:00,False,True
4,lama__canonical__canonical__p002_loss_large,lama,lama,NaN,iopaint_lama,iopaint_lama,model_inference,restoration_eval.restoration_lama,2.0.0,cuda,...,NaN,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/16_lama_difference_maps/figures/all_ca...,ok,16_lama_difference_maps,1.0.0,restoration_eval.error_maps,2026-07-28T12:19:32.929592+00:00,False,True


In [34]:
case_asset_required_columns = {
    "notebook",
    "spatial_schema_version",
    "implementation",
    "restoration_case_id",
    "dataset_name",
    "case_id",
    "painting_id",
    "model_name",
    "clean_path_project_relative",
    "damaged_path_project_relative",
    "mask_path_project_relative",
    "restored_path_project_relative",
    "diagnostic_status",
    "selected_figure_count",
    "has_selected_figure",
    "has_all_case_figure",
}

missing_case_asset_columns = sorted(
    case_asset_required_columns - set(case_asset_index_df.columns)
)

case_asset_validation_df = pd.DataFrame(
    [
        build_check(
            "case_asset_rows",
            len(case_asset_index_df),
            len(lama_ready_cases_df),
            len(case_asset_index_df) == len(lama_ready_cases_df),
        ),
        build_check(
            "case_asset_required_columns",
            missing_case_asset_columns,
            [],
            not missing_case_asset_columns,
        ),
        build_check(
            "case_asset_unique_restoration_ids",
            int(case_asset_index_df["restoration_case_id"].duplicated().sum()),
            0,
            not case_asset_index_df["restoration_case_id"].duplicated().any(),
        ),
        build_check(
            "case_asset_diagnostics_ok",
            int(case_asset_index_df["diagnostic_status"].eq("ok").sum()),
            len(case_asset_index_df),
            bool(case_asset_index_df["diagnostic_status"].eq("ok").all()),
        ),
        build_check(
            "case_asset_selected_figures_present",
            int(case_asset_index_df["has_selected_figure"].sum()),
            "> 0",
            bool(case_asset_index_df["has_selected_figure"].any()),
        ),
    ]
)

display(case_asset_validation_df)

if not case_asset_validation_df["passed"].astype(bool).all():
    raise RuntimeError("Case asset index validation failed.")

case_asset_index_df.to_csv(CASE_ASSETS_OUTPUT_PATH, index=False)

case_asset_readback_df = pd.read_csv(CASE_ASSETS_OUTPUT_PATH)

case_asset_readback_checks_df = pd.DataFrame(
    [
        build_check(
            "case_asset_output_exists",
            CASE_ASSETS_OUTPUT_PATH.is_file(),
            True,
            CASE_ASSETS_OUTPUT_PATH.is_file(),
        ),
        build_check(
            "case_asset_readback_rows",
            len(case_asset_readback_df),
            len(case_asset_index_df),
            len(case_asset_readback_df) == len(case_asset_index_df),
        ),
    ]
)

display(case_asset_readback_checks_df)

if not case_asset_readback_checks_df["passed"].astype(bool).all():
    raise RuntimeError("Case asset index readback validation failed.")

print("Saved case asset index:", project_relative_path(CASE_ASSETS_OUTPUT_PATH))
print("Case asset rows:", len(case_asset_index_df))

,check,observed,expected,passed
0,case_asset_rows,360,360,True
1,case_asset_required_columns,[],[],True
2,case_asset_unique_restoration_ids,0,0,True
3,case_asset_diagnostics_ok,360,360,True
4,case_asset_selected_figures_present,37,> 0,True


,check,observed,expected,passed
0,case_asset_output_exists,True,True,True
1,case_asset_readback_rows,360,360,True


Saved case asset index: data/processed/metrics/spatial_diagnostic_case_assets_lama.csv
Case asset rows: 360


In [35]:
case_asset_required_columns = {
    "notebook",
    "spatial_schema_version",
    "implementation",
    "restoration_case_id",
    "dataset_name",
    "case_id",
    "painting_id",
    "model_name",
    "clean_path_project_relative",
    "damaged_path_project_relative",
    "mask_path_project_relative",
    "restored_path_project_relative",
    "diagnostic_status",
    "selected_figure_count",
    "has_selected_figure",
    "has_all_case_figure",
}

missing_case_asset_columns = sorted(
    case_asset_required_columns - set(case_asset_index_df.columns)
)

case_asset_validation_df = pd.DataFrame(
    [
        build_check(
            "case_asset_rows",
            len(case_asset_index_df),
            len(lama_ready_cases_df),
            len(case_asset_index_df) == len(lama_ready_cases_df),
        ),
        build_check(
            "case_asset_required_columns",
            missing_case_asset_columns,
            [],
            not missing_case_asset_columns,
        ),
        build_check(
            "case_asset_unique_restoration_ids",
            int(case_asset_index_df["restoration_case_id"].duplicated().sum()),
            0,
            not case_asset_index_df["restoration_case_id"].duplicated().any(),
        ),
        build_check(
            "case_asset_diagnostics_ok",
            int(case_asset_index_df["diagnostic_status"].eq("ok").sum()),
            len(case_asset_index_df),
            bool(case_asset_index_df["diagnostic_status"].eq("ok").all()),
        ),
        build_check(
            "case_asset_selected_figures_present",
            int(case_asset_index_df["has_selected_figure"].sum()),
            "> 0",
            bool(case_asset_index_df["has_selected_figure"].any()),
        ),
    ]
)

display(case_asset_validation_df)

if not case_asset_validation_df["passed"].astype(bool).all():
    raise RuntimeError("Case asset index validation failed.")

case_asset_index_df.to_csv(CASE_ASSETS_OUTPUT_PATH, index=False)

case_asset_readback_df = pd.read_csv(CASE_ASSETS_OUTPUT_PATH)

case_asset_readback_checks_df = pd.DataFrame(
    [
        build_check(
            "case_asset_output_exists",
            CASE_ASSETS_OUTPUT_PATH.is_file(),
            True,
            CASE_ASSETS_OUTPUT_PATH.is_file(),
        ),
        build_check(
            "case_asset_readback_rows",
            len(case_asset_readback_df),
            len(case_asset_index_df),
            len(case_asset_readback_df) == len(case_asset_index_df),
        ),
    ]
)

display(case_asset_readback_checks_df)

if not case_asset_readback_checks_df["passed"].astype(bool).all():
    raise RuntimeError("Case asset index readback validation failed.")

print("Saved case asset index:", project_relative_path(CASE_ASSETS_OUTPUT_PATH))
print("Case asset rows:", len(case_asset_index_df))

,check,observed,expected,passed
0,case_asset_rows,360,360,True
1,case_asset_required_columns,[],[],True
2,case_asset_unique_restoration_ids,0,0,True
3,case_asset_diagnostics_ok,360,360,True
4,case_asset_selected_figures_present,37,> 0,True


,check,observed,expected,passed
0,case_asset_output_exists,True,True,True
1,case_asset_readback_rows,360,360,True


Saved case asset index: data/processed/metrics/spatial_diagnostic_case_assets_lama.csv
Case asset rows: 360


In [37]:
final_artifact_rows = [
    {
        "artifact_key": "spatial_diagnostics",
        "artifact_label": "Canonical LaMa spatial diagnostics",
        "path": SPATIAL_DIAGNOSTICS_OUTPUT_PATH,
        "required": True,
        "artifact_type": "csv",
    },
    {
        "artifact_key": "visualization_scales",
        "artifact_label": "LaMa visualization scales",
        "path": VISUALIZATION_SCALES_OUTPUT_PATH,
        "required": True,
        "artifact_type": "csv",
    },
    {
        "artifact_key": "case_assets",
        "artifact_label": "LaMa spatial diagnostic case asset index",
        "path": CASE_ASSETS_OUTPUT_PATH,
        "required": True,
        "artifact_type": "csv",
    },
    {
        "artifact_key": "selected_figure_manifest",
        "artifact_label": "Selected LaMa difference-map figure manifest",
        "path": SELECTED_FIGURE_MANIFEST_PATH,
        "required": True,
        "artifact_type": "csv",
    },
    {
        "artifact_key": "all_figure_manifest",
        "artifact_label": "All-case LaMa difference-map figure manifest",
        "path": ALL_FIGURE_MANIFEST_PATH,
        "required": bool(GENERATE_ALL_CASE_FIGURES),
        "artifact_type": "csv",
    },
]

final_artifact_inventory_df = pd.DataFrame(final_artifact_rows)

final_artifact_inventory_df["exists"] = final_artifact_inventory_df["path"].map(
    lambda path_value: Path(path_value).is_file()
)
final_artifact_inventory_df["path_project_relative"] = final_artifact_inventory_df[
    "path"
].map(project_relative_path)
final_artifact_inventory_df["file_size_bytes"] = final_artifact_inventory_df["path"].map(
    lambda path_value: Path(path_value).stat().st_size if Path(path_value).is_file() else 0
)

display(final_artifact_inventory_df)

,artifact_key,artifact_label,path,required,artifact_type,exists,path_project_relative,file_size_bytes
0,spatial_diagnostics,Canonical LaMa spatial diagnostics,D:\Masters\FH\Thesis\painting-restoration-eval...,True,csv,True,data/processed/metrics/spatial_diagnostics_lam...,618205
1,visualization_scales,LaMa visualization scales,D:\Masters\FH\Thesis\painting-restoration-eval...,True,csv,True,data/processed/metrics/spatial_diagnostic_scal...,753
2,case_assets,LaMa spatial diagnostic case asset index,D:\Masters\FH\Thesis\painting-restoration-eval...,True,csv,True,data/processed/metrics/spatial_diagnostic_case...,1404903
3,selected_figure_manifest,Selected LaMa difference-map figure manifest,D:\Masters\FH\Thesis\painting-restoration-eval...,True,csv,True,outputs/16_lama_difference_maps/lama_differenc...,78370
4,all_figure_manifest,All-case LaMa difference-map figure manifest,D:\Masters\FH\Thesis\painting-restoration-eval...,True,csv,True,outputs/16_lama_difference_maps/lama_differenc...,599615


In [38]:
final_readback_tables = {}

for _, artifact_row in final_artifact_inventory_df.iterrows():
    artifact_key = artifact_row["artifact_key"]
    artifact_path = Path(artifact_row["path"])

    if not artifact_path.is_file():
        final_readback_tables[artifact_key] = pd.DataFrame()
        continue

    if artifact_row["artifact_type"] == "csv":
        final_readback_tables[artifact_key] = pd.read_csv(artifact_path)
    else:
        final_readback_tables[artifact_key] = pd.DataFrame()

final_readback_summary_df = pd.DataFrame(
    [
        {
            "artifact_key": artifact_key,
            "rows": len(table_df),
            "columns": len(table_df.columns),
        }
        for artifact_key, table_df in final_readback_tables.items()
    ]
).sort_values("artifact_key")

display(final_readback_summary_df)

,artifact_key,rows,columns
4,all_figure_manifest,360,109
2,case_assets,360,259
3,selected_figure_manifest,40,111
0,spatial_diagnostics,360,112
1,visualization_scales,2,21


In [39]:
spatial_diagnostics_readback_df = final_readback_tables["spatial_diagnostics"]
visualization_scales_readback_df = final_readback_tables["visualization_scales"]
case_assets_readback_df = final_readback_tables["case_assets"]
selected_manifest_readback_df = final_readback_tables["selected_figure_manifest"]
all_manifest_readback_df = final_readback_tables["all_figure_manifest"]

required_artifacts_missing = final_artifact_inventory_df.loc[
    final_artifact_inventory_df["required"] & ~final_artifact_inventory_df["exists"],
    "artifact_key",
].tolist()

final_validation_checks = [
    build_check(
        "required_artifacts_exist",
        required_artifacts_missing,
        [],
        not required_artifacts_missing,
    ),
    build_check(
        "spatial_diagnostics_rows",
        len(spatial_diagnostics_readback_df),
        len(lama_ready_cases_df),
        len(spatial_diagnostics_readback_df) == len(lama_ready_cases_df),
    ),
    build_check(
        "spatial_diagnostics_status_ok",
        int(spatial_diagnostics_readback_df["diagnostic_status"].eq("ok").sum()),
        len(spatial_diagnostics_readback_df),
        bool(spatial_diagnostics_readback_df["diagnostic_status"].eq("ok").all()),
    ),
    build_check(
        "visualization_scale_rows",
        len(visualization_scales_readback_df),
        2,
        len(visualization_scales_readback_df) == 2,
    ),
    build_check(
        "case_asset_rows",
        len(case_assets_readback_df),
        len(lama_ready_cases_df),
        len(case_assets_readback_df) == len(lama_ready_cases_df),
    ),
    build_check(
        "case_asset_selected_figures_present",
        int(case_assets_readback_df["has_selected_figure"].sum()),
        "> 0",
        bool(case_assets_readback_df["has_selected_figure"].any()),
    ),
    build_check(
        "selected_manifest_nonempty",
        len(selected_manifest_readback_df),
        "> 0",
        len(selected_manifest_readback_df) > 0,
    ),
    build_check(
        "selected_manifest_status_ok",
        int(selected_manifest_readback_df["status"].eq("ok").sum()),
        len(selected_manifest_readback_df),
        bool(selected_manifest_readback_df["status"].eq("ok").all()),
    ),
]

if GENERATE_ALL_CASE_FIGURES:
    final_validation_checks.extend(
        [
            build_check(
                "all_manifest_nonempty",
                len(all_manifest_readback_df),
                "> 0",
                len(all_manifest_readback_df) > 0,
            ),
            build_check(
                "all_manifest_status_ok",
                int(all_manifest_readback_df["status"].eq("ok").sum()),
                len(all_manifest_readback_df),
                bool(all_manifest_readback_df["status"].eq("ok").all()),
            ),
        ]
    )

final_validation_df = pd.DataFrame(final_validation_checks)
final_validation_df.insert(0, "notebook", NOTEBOOK_NAME)
final_validation_df.insert(1, "model_name", MODEL_NAME)
final_validation_df.insert(2, "spatial_schema_version", SPATIAL_SCHEMA_VERSION)
final_validation_df["created_at_utc"] = datetime.now(timezone.utc).isoformat()

display(final_validation_df)

if not final_validation_df["passed"].astype(bool).all():
    raise RuntimeError("Notebook 16 final validation failed.")

,notebook,model_name,spatial_schema_version,check,observed,expected,passed,created_at_utc
0,16_lama_difference_maps,lama,1.0.0,required_artifacts_exist,[],[],True,2026-07-28T12:25:08.507129+00:00
1,16_lama_difference_maps,lama,1.0.0,spatial_diagnostics_rows,360,360,True,2026-07-28T12:25:08.507129+00:00
2,16_lama_difference_maps,lama,1.0.0,spatial_diagnostics_status_ok,360,360,True,2026-07-28T12:25:08.507129+00:00
3,16_lama_difference_maps,lama,1.0.0,visualization_scale_rows,2,2,True,2026-07-28T12:25:08.507129+00:00
4,16_lama_difference_maps,lama,1.0.0,case_asset_rows,360,360,True,2026-07-28T12:25:08.507129+00:00
5,16_lama_difference_maps,lama,1.0.0,case_asset_selected_figures_present,37,> 0,True,2026-07-28T12:25:08.507129+00:00
6,16_lama_difference_maps,lama,1.0.0,selected_manifest_nonempty,40,> 0,True,2026-07-28T12:25:08.507129+00:00
7,16_lama_difference_maps,lama,1.0.0,selected_manifest_status_ok,40,40,True,2026-07-28T12:25:08.507129+00:00
8,16_lama_difference_maps,lama,1.0.0,all_manifest_nonempty,360,> 0,True,2026-07-28T12:25:08.507129+00:00
9,16_lama_difference_maps,lama,1.0.0,all_manifest_status_ok,360,360,True,2026-07-28T12:25:08.507129+00:00


In [40]:
final_validation_df.to_csv(SPATIAL_VALIDATION_OUTPUT_PATH, index=False)

final_validation_readback_df = pd.read_csv(SPATIAL_VALIDATION_OUTPUT_PATH)

final_validation_readback_checks_df = pd.DataFrame(
    [
        build_check(
            "final_validation_output_exists",
            SPATIAL_VALIDATION_OUTPUT_PATH.is_file(),
            True,
            SPATIAL_VALIDATION_OUTPUT_PATH.is_file(),
        ),
        build_check(
            "final_validation_readback_rows",
            len(final_validation_readback_df),
            len(final_validation_df),
            len(final_validation_readback_df) == len(final_validation_df),
        ),
        build_check(
            "final_validation_readback_passed",
            int(final_validation_readback_df["passed"].astype(bool).sum()),
            len(final_validation_readback_df),
            bool(final_validation_readback_df["passed"].astype(bool).all()),
        ),
    ]
)

display(final_validation_readback_checks_df)

if not final_validation_readback_checks_df["passed"].astype(bool).all():
    raise RuntimeError("Final validation readback failed.")

print("Saved final validation CSV:", project_relative_path(SPATIAL_VALIDATION_OUTPUT_PATH))

,check,observed,expected,passed
0,final_validation_output_exists,True,True,True
1,final_validation_readback_rows,10,10,True
2,final_validation_readback_passed,10,10,True


Saved final validation CSV: outputs/16_lama_difference_maps/lama_difference_map_validation.csv


In [41]:
stage_manifest = {
    "notebook": NOTEBOOK_NAME,
    "model_name": MODEL_NAME,
    "spatial_schema_version": SPATIAL_SCHEMA_VERSION,
    "implementation": SPATIAL_IMPLEMENTATION_NAME,
    "run_started_at_utc": RUN_STARTED_AT_UTC,
    "manifest_created_at_utc": datetime.now(timezone.utc).isoformat(),
    "configuration": {
        "target_size": TARGET_SIZE,
        "mask_binary_threshold": MASK_BINARY_THRESHOLD,
        "boundary_width_pixels": BOUNDARY_WIDTH_PIXELS,
        "boundary_mode": BOUNDARY_MODE,
        "absolute_error_percentile": ABSOLUTE_ERROR_PERCENTILE,
        "signed_improvement_percentile": SIGNED_IMPROVEMENT_PERCENTILE,
        "scale_sample_region": SCALE_SAMPLE_REGION,
        "generate_all_case_figures": GENERATE_ALL_CASE_FIGURES,
        "include_zero_control_figures": INCLUDE_ZERO_CONTROL_FIGURES,
        "figure_dpi": FIGURE_DPI,
    },
    "summary": {
        "ready_case_count": int(len(lama_ready_cases_df)),
        "spatial_diagnostic_rows": int(len(spatial_diagnostics_readback_df)),
        "case_asset_rows": int(len(case_assets_readback_df)),
        "selected_figure_rows": int(len(selected_manifest_readback_df)),
        "all_case_figure_rows": int(len(all_manifest_readback_df)),
        "selected_unique_cases": int(
            selected_manifest_readback_df["restoration_case_id"].nunique()
            if "restoration_case_id" in selected_manifest_readback_df.columns
            else 0
        ),
        "validation_checks": int(len(final_validation_df)),
        "validation_passed": bool(final_validation_df["passed"].astype(bool).all()),
    },
    "artifacts": {
        row["artifact_key"]: {
            "label": row["artifact_label"],
            "type": row["artifact_type"],
            "required": bool(row["required"]),
            "exists": bool(row["exists"]),
            "path": row["path_project_relative"],
            "file_size_bytes": int(row["file_size_bytes"]),
        }
        for _, row in final_artifact_inventory_df.iterrows()
    },
    "validation": {
        "path": project_relative_path(SPATIAL_VALIDATION_OUTPUT_PATH),
        "passed": bool(final_validation_df["passed"].astype(bool).all()),
    },
}

STAGE_MANIFEST_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)

with STAGE_MANIFEST_JSON_PATH.open("w", encoding="utf-8") as manifest_file:
    json.dump(stage_manifest, manifest_file, indent=2)

with STAGE_MANIFEST_JSON_PATH.open("r", encoding="utf-8") as manifest_file:
    stage_manifest_readback = json.load(manifest_file)

stage_manifest_checks_df = pd.DataFrame(
    [
        build_check(
            "stage_manifest_output_exists",
            STAGE_MANIFEST_JSON_PATH.is_file(),
            True,
            STAGE_MANIFEST_JSON_PATH.is_file(),
        ),
        build_check(
            "stage_manifest_validation_passed",
            stage_manifest_readback["validation"]["passed"],
            True,
            bool(stage_manifest_readback["validation"]["passed"]),
        ),
        build_check(
            "stage_manifest_ready_case_count",
            stage_manifest_readback["summary"]["ready_case_count"],
            len(lama_ready_cases_df),
            int(stage_manifest_readback["summary"]["ready_case_count"]) == len(lama_ready_cases_df),
        ),
    ]
)

display(stage_manifest_checks_df)

if not stage_manifest_checks_df["passed"].astype(bool).all():
    raise RuntimeError("Stage manifest validation failed.")

print("Saved stage manifest JSON:", project_relative_path(STAGE_MANIFEST_JSON_PATH))

,check,observed,expected,passed
0,stage_manifest_output_exists,True,True,True
1,stage_manifest_validation_passed,True,True,True
2,stage_manifest_ready_case_count,360,360,True


Saved stage manifest JSON: outputs/reports/16_lama_difference_maps/lama_difference_maps_manifest.json


In [42]:
completion_summary_df = pd.DataFrame(
    [
        {
            "output": "spatial_diagnostics",
            "path": project_relative_path(SPATIAL_DIAGNOSTICS_OUTPUT_PATH),
            "rows": len(spatial_diagnostics_readback_df),
        },
        {
            "output": "visualization_scales",
            "path": project_relative_path(VISUALIZATION_SCALES_OUTPUT_PATH),
            "rows": len(visualization_scales_readback_df),
        },
        {
            "output": "case_assets",
            "path": project_relative_path(CASE_ASSETS_OUTPUT_PATH),
            "rows": len(case_assets_readback_df),
        },
        {
            "output": "selected_figure_manifest",
            "path": project_relative_path(SELECTED_FIGURE_MANIFEST_PATH),
            "rows": len(selected_manifest_readback_df),
        },
        {
            "output": "all_figure_manifest",
            "path": project_relative_path(ALL_FIGURE_MANIFEST_PATH),
            "rows": len(all_manifest_readback_df),
        },
        {
            "output": "final_validation",
            "path": project_relative_path(SPATIAL_VALIDATION_OUTPUT_PATH),
            "rows": len(final_validation_df),
        },
        {
            "output": "stage_manifest_json",
            "path": project_relative_path(STAGE_MANIFEST_JSON_PATH),
            "rows": 1,
        },
    ]
)

display(completion_summary_df)

print("Notebook 16 complete.")
print("Final validation passed:", bool(final_validation_df["passed"].astype(bool).all()))

,output,path,rows
0,spatial_diagnostics,data/processed/metrics/spatial_diagnostics_lam...,360
1,visualization_scales,data/processed/metrics/spatial_diagnostic_scal...,2
2,case_assets,data/processed/metrics/spatial_diagnostic_case...,360
3,selected_figure_manifest,outputs/16_lama_difference_maps/lama_differenc...,40
4,all_figure_manifest,outputs/16_lama_difference_maps/lama_differenc...,360
5,final_validation,outputs/16_lama_difference_maps/lama_differenc...,10
6,stage_manifest_json,outputs/reports/16_lama_difference_maps/lama_d...,1


Notebook 16 complete.
Final validation passed: True
